In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 設定（ここを変えてA/Bテスト）
# ============================================================
MIXUP_N_2WAY     = 2000    # 2-way mixup生成数 (元: 500)
MIXUP_N_3WAY     = 1000    # 3-way mixup生成数 (新規)
MIXUP_ALPHA_2WAY = 0.3     # Beta分布α (試す値: 0.2, 0.3, 0.5, 1.0)
MIXUP_ALPHA_3WAY = 0.5     # Dirichlet α
N_ENSEMBLE_SEEDS = 5       # Multi-seed数 (1なら従来と同じ)
AUG_SAMPLE_WEIGHT = 0.5    # 拡張データの学習重み (1.0=等重み)

print("=" * 60)
print(f"🧪 Multi-Seed Mixup Ensemble")
print(f"   2-way: {MIXUP_N_2WAY} samples (α={MIXUP_ALPHA_2WAY})")
print(f"   3-way: {MIXUP_N_3WAY} samples (α={MIXUP_ALPHA_3WAY})")
print(f"   Seeds: {N_ENSEMBLE_SEEDS}")
print(f"   Aug weight: {AUG_SAMPLE_WEIGHT}")
print("=" * 60)

# ============================================================
# 過去スコア記録
# ============================================================
HISTORY = [
    ("LGB単独(元特徴量)",       17.21, None,  12.615),
    ("元Blend(LGB/PLS/Ridge)", 14.10, None,  12.647),
    ("正則化強化(LGB単独)",     17.60, None,  12.760),
    ("Huber Loss",             17.53, None,  12.770),
    ("PLS予測を特徴量追加",     15.65, None,  12.800),
    ("逆距離加重KNN",          17.13, None,  12.940),
    ("d2(二次微分)追加",        16.12, None,  13.410),
    ("物理特徴量53個追加",      12.68, None,  14.500),
    ("Mixup(500, α=0.3)",     None,  None,  11.80),
]

# ============================================================
# 1. データ読み込み
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit = pd.read_csv('data/sample_submit.csv', header=None)

train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)
idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))


def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s


# ============================================================
# 2. Augmentation関数
# ============================================================
def mixup_2way(X, y, species, n_augment, alpha, rng):
    """異なる2樹種のスペクトルを線形補間"""
    unique_sp = np.unique(species)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        sp1, sp2 = rng.choice(unique_sp, size=2, replace=False)
        i1 = rng.choice(np.where(species == sp1)[0])
        i2 = rng.choice(np.where(species == sp2)[0])
        lam = rng.beta(alpha, alpha)
        X_aug.append(lam * X[i1] + (1 - lam) * X[i2])
        y_aug.append(lam * y[i1] + (1 - lam) * y[i2])
    return np.array(X_aug), np.array(y_aug)


def mixup_3way(X, y, species, n_augment, alpha, rng):
    """異なる3樹種のスペクトルをDirichlet重みで混合"""
    unique_sp = np.unique(species)
    if len(unique_sp) < 3:
        return np.empty((0, X.shape[1])), np.empty(0)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        sps = rng.choice(unique_sp, size=3, replace=False)
        idxs = [rng.choice(np.where(species == sp)[0]) for sp in sps]
        weights = rng.dirichlet([alpha] * 3)
        x_mix = sum(w * X[i] for w, i in zip(weights, idxs))
        y_mix = sum(w * y[i] for w, i in zip(weights, idxs))
        X_aug.append(x_mix)
        y_aug.append(y_mix)
    return np.array(X_aug), np.array(y_aug)


# ============================================================
# 3. 特徴量作成
# ============================================================
def make_features(X_raw):
    snv = apply_snv(X_raw)
    d1 = savgol_filter(snv, window_length=15, polyorder=2, deriv=1, axis=1)
    ratio = (X_raw[:, idx_1940] / (X_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
    std = np.std(X_raw, axis=1, keepdims=True)
    return snv, d1, ratio, std


# ============================================================
# 4. Multi-Seed Ensemble CV
# ============================================================
gkf = GroupKFold(n_splits=5)
X_test_raw = test[spec_cols].values

all_seed_test = []
all_seed_oof = []

for seed_idx in range(N_ENSEMBLE_SEEDS):
    base_seed = seed_idx * 1000
    print(f"\n{'='*60}")
    print(f"🌱 Seed {seed_idx+1}/{N_ENSEMBLE_SEEDS} (base={base_seed})")
    print(f"{'='*60}")

    final_lgb = np.zeros(len(test))
    oof_lgb = np.zeros(len(train))
    fold_rmses = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        train[spec_cols].values, y_train_log, groups
    )):
        va_species = train.iloc[va_idx]['樹種'].unique()
        tr_sp_nums = groups.iloc[tr_idx].values

        X_tr_raw = train[spec_cols].values[tr_idx]
        y_tr = y_train_log.iloc[tr_idx].values
        X_va_raw = train[spec_cols].values[va_idx]
        y_va = y_train_log.iloc[va_idx].values

        # ── Augmentation ──
        fold_seed = base_seed + fold
        rng = np.random.RandomState(fold_seed)

        X_m2, y_m2 = mixup_2way(X_tr_raw, y_tr, tr_sp_nums,
                                  MIXUP_N_2WAY, MIXUP_ALPHA_2WAY, rng)
        X_m3, y_m3 = mixup_3way(X_tr_raw, y_tr, tr_sp_nums,
                                  MIXUP_N_3WAY, MIXUP_ALPHA_3WAY, rng)

        n_orig = len(X_tr_raw)
        X_tr_aug = np.vstack([X_tr_raw, X_m2, X_m3])
        y_tr_aug = np.concatenate([y_tr, y_m2, y_m3])

        # Sample weights（実データ優先）
        w = np.ones(len(y_tr_aug))
        w[n_orig:] = AUG_SAMPLE_WEIGHT

        if fold == 0 and seed_idx == 0:
            print(f"  元: {n_orig}, 2way: {len(X_m2)}, 3way: {len(X_m3)}")
            print(f"  合計: {len(X_tr_aug)} (aug weight={AUG_SAMPLE_WEIGHT})")

        # ── 特徴量 ──
        snv_tr, d1_tr, ratio_tr, std_tr = make_features(X_tr_aug)
        snv_va, d1_va, ratio_va, std_va = make_features(X_va_raw)
        snv_te, d1_te, ratio_te, std_te = make_features(X_test_raw)

        # PCA（元データのみでfit）
        snv_tr_orig = apply_snv(X_tr_raw)
        pca = PCA(n_components=10, random_state=42)
        pca.fit(snv_tr_orig)
        pca_tr = pca.transform(snv_tr)
        pca_va = pca.transform(snv_va)
        pca_te = pca.transform(snv_te)

        # KNN（元データのみでfit）
        pca_tr_orig = pca.transform(snv_tr_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='cosine')
        knn.fit(pca_tr_orig)

        # KNN特徴量: train
        _, ind_tr = knn.kneighbors(pca_tr, n_neighbors=6)
        knn_ymean_tr = np.zeros(len(X_tr_aug))
        for i in range(len(X_tr_aug)):
            nb = ind_tr[i]
            if i < n_orig:
                valid = nb[nb != i][:5]
            else:
                valid = nb[:5]
            knn_ymean_tr[i] = np.mean(y_tr[valid])
        knn_ymean_tr = knn_ymean_tr.reshape(-1, 1)

        # KNN特徴量: val / test
        _, ind_va = knn.kneighbors(pca_va, n_neighbors=5)
        knn_ymean_va = np.mean(y_tr[ind_va], axis=1).reshape(-1, 1)
        _, ind_te = knn.kneighbors(pca_te, n_neighbors=5)
        knn_ymean_te = np.mean(y_tr[ind_te], axis=1).reshape(-1, 1)

        # 特徴量結合
        feat_tr = np.hstack([snv_tr, d1_tr, pca_tr, knn_ymean_tr, ratio_tr, std_tr])
        feat_va = np.hstack([snv_va, d1_va, pca_va, knn_ymean_va, ratio_va, std_va])
        feat_te = np.hstack([snv_te, d1_te, pca_te, knn_ymean_te, ratio_te, std_te])

        # ── LightGBM ──
        lgb_model = lgb.LGBMRegressor(
            n_estimators=1000, learning_rate=0.03,
            max_depth=5, num_leaves=31,
            subsample=0.8, colsample_bytree=0.3,
            min_child_samples=20,
            random_state=42 + seed_idx,
            verbosity=-1
        )
        lgb_model.fit(
            feat_tr, y_tr_aug,
            sample_weight=w,
            eval_set=[(feat_va, y_va)],
            callbacks=[lgb.early_stopping(30, verbose=False)]
        )

        p_va = np.expm1(lgb_model.predict(feat_va))
        p_te = np.expm1(lgb_model.predict(feat_te))
        oof_lgb[va_idx] = p_va
        final_lgb += p_te / 5

        rmse = np.sqrt(mean_squared_error(np.expm1(y_va), p_va))
        fold_rmses.append(rmse)
        print(f"  Fold {fold+1} RMSE: {rmse:.4f}  (valid species: {list(va_species)})")

    oof_rmse = np.sqrt(mean_squared_error(np.expm1(y_train_log), oof_lgb))
    fold_mean = np.mean(fold_rmses)
    fold_std = np.std(fold_rmses)
    print(f"  🌟 Seed {seed_idx+1} OOF RMSE: {oof_rmse:.4f} "
          f"(Fold平均: {fold_mean:.4f} ± {fold_std:.4f})")

    all_seed_test.append(final_lgb)
    all_seed_oof.append(oof_lgb)


# ============================================================
# 5. Ensemble集約
# ============================================================
final_ensemble = np.mean(all_seed_test, axis=0)
oof_ensemble = np.mean(all_seed_oof, axis=0)

y_true = np.expm1(y_train_log)
oof_rmse_ens = np.sqrt(mean_squared_error(y_true, oof_ensemble))

print(f"\n{'='*60}")
print(f"📊 Ensemble結果 ({N_ENSEMBLE_SEEDS} seeds)")
print(f"{'='*60}")
print(f"  🌟 Ensemble OOF RMSE: {oof_rmse_ens:.4f}")

# 個別seed間のばらつき
for i, pred in enumerate(all_seed_test):
    diff = np.sqrt(np.mean((pred - final_ensemble)**2))
    print(f"  Seed {i+1} vs Ensemble RMSD: {diff:.4f}")

# 樹種別
print(f"\n  {'樹種':12s} {'n':>4s} {'RMSE':>7s} {'bias':>7s}")
for sp in sorted(train['樹種'].unique()):
    mask = train['樹種'] == sp
    y_s = train.loc[mask, '含水率'].values
    p_s = oof_ensemble[mask.values]
    r = np.sqrt(np.mean((y_s - p_s)**2))
    b = np.mean(y_s - p_s)
    print(f"  {sp:12s} {len(y_s):4d} {r:7.2f} {b:+7.2f}")


# ============================================================
# 6. 提出
# ============================================================
final_out = np.clip(final_ensemble, 0, None)
submit[1] = final_out
fname = (f'submission_multiseed{N_ENSEMBLE_SEEDS}'
         f'_2way{MIXUP_N_2WAY}_3way{MIXUP_N_3WAY}.csv')
submit.to_csv(fname, index=False, header=False)

print(f"\n✅ 提出ファイル: {fname}")
print(f"📈 予測分布: min={final_out.min():.1f}%, "
      f"median={np.median(final_out):.1f}%, "
      f"max={final_out.max():.1f}%")


# ============================================================
# 7. 過去スコア一覧 + 今回の結果
# ============================================================
print(f"\n{'='*60}")
print("📌 全スコア比較（過去 → 今回）")
print(f"{'='*60}")
print(f"  {'手法':<30s} {'OOF RMSE':>10s} {'Fold平均':>10s} {'LB RMSE':>10s}")
print(f"  {'─'*30} {'─'*10} {'─'*10} {'─'*10}")

for name, oof, fold_avg, lb in HISTORY:
    oof_str = f"{oof:.2f}" if oof is not None else "---"
    fold_str = f"{fold_avg:.2f}" if fold_avg is not None else "---"
    lb_str = f"{lb:.3f}" if lb is not None else "---"
    marker = " ← 前BEST" if lb is not None and lb == 11.80 else ""
    print(f"  {name:<30s} {oof_str:>10s} {fold_str:>10s} {lb_str:>10s}{marker}")

# 今回のスコアを表示
today_name = (f"MultiSeed{N_ENSEMBLE_SEEDS}_2w{MIXUP_N_2WAY}_3w{MIXUP_N_3WAY}")
print(f"  {today_name:<30s} {oof_rmse_ens:>10.2f} {'---':>10s} {'???':>10s} ← 今回")

print(f"\n{'='*60}")
print("📊 次のアクション候補")
print(f"{'='*60}")
print("""
  LB改善した場合:
    → α値を変更 (2way: 0.2/0.5, 3way: 0.3/1.0)
    → 生成数を変更 (2way: 3000, 3way: 2000)
    → AUG_SAMPLE_WEIGHT調整 (0.3, 0.7, 1.0)
    
  LB悪化した場合:
    → 3wayを除外 (2wayのみに戻す)
    → 生成数を減らす (2way: 500, 3way: 0)
    → Multi-seed数を減らす (3 seeds)
    → α値を小さく (0.1) → 元データに近い混合
""")

🧪 Multi-Seed Mixup Ensemble
   2-way: 2000 samples (α=0.3)
   3-way: 1000 samples (α=0.5)
   Seeds: 5
   Aug weight: 0.5

🌱 Seed 1/5 (base=0)
  元: 940, 2way: 2000, 3way: 1000
  合計: 3940 (aug weight=0.5)


In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIG - A/B切り替え
# ============================================================
USE_CONFIG_B = False  # True → d2一本化（攻め）, False → 元構成（安全）

MIXUP_N       = 500     # 元と同じ
MIXUP_ALPHA   = 0.3     # 元と同じ
AUG_WEIGHT    = 1.0     # 元と同じ（等重み）
ENSEMBLE_SEEDS = [42, 100, 200]  # seed42=元の11.80を含む

config_name = "B_d2only" if USE_CONFIG_B else "A_safe_multiseed"

print("=" * 60)
print(f"🧪 Config: {config_name}")
print(f"   Mixup: {MIXUP_N} samples, α={MIXUP_ALPHA}")
print(f"   Seeds: {ENSEMBLE_SEEDS}")
print(f"   d2 only: {USE_CONFIG_B}")
print("=" * 60)

# ============================================================
# 過去スコア記録
# ============================================================
HISTORY = [
    ("LGB単独(元特徴量)",       17.21, None,  12.615),
    ("元Blend(LGB/PLS/Ridge)", 14.10, None,  12.647),
    ("正則化強化(LGB単独)",     17.60, None,  12.760),
    ("Huber Loss",             17.53, None,  12.770),
    ("PLS予測を特徴量追加",     15.65, None,  12.800),
    ("逆距離加重KNN",          17.13, None,  12.940),
    ("d2(二次微分)追加",        16.12, None,  13.410),
    ("物理特徴量53個追加",      12.68, None,  14.500),
    ("Mixup(500,α=0.3)",      None,  None,  11.80),
    ("MultiSeed5+2w2000+3w1000", None, None, 12.249),
]

# ============================================================
# 1. データ読み込み
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit = pd.read_csv('data/sample_submit.csv', header=None)

train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)
idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))


def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s


# ============================================================
# 2. Mixup（元と完全に同じ）
# ============================================================
def mixup_2way(X, y, species, n_augment, alpha, rng):
    unique_sp = np.unique(species)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        sp1, sp2 = rng.choice(unique_sp, size=2, replace=False)
        i1 = rng.choice(np.where(species == sp1)[0])
        i2 = rng.choice(np.where(species == sp2)[0])
        lam = rng.beta(alpha, alpha)
        X_aug.append(lam * X[i1] + (1 - lam) * X[i2])
        y_aug.append(lam * y[i1] + (1 - lam) * y[i2])
    return np.array(X_aug), np.array(y_aug)


# ============================================================
# 3. 特徴量作成（A/B分岐）
# ============================================================
def make_features_A(X_raw):
    """Config A: 元コードと完全同一（SNV + d1）"""
    snv = apply_snv(X_raw)
    d1 = savgol_filter(snv, window_length=15, polyorder=2, deriv=1, axis=1)
    ratio = (X_raw[:, idx_1940] / (X_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
    std = np.std(X_raw, axis=1, keepdims=True)
    return snv, d1, ratio, std


def make_features_B(X_raw):
    """Config B: d2一本化（SNV+d1を置換→次元半減）"""
    # 2次微分のみ（ベースライン完全除去、散乱補正不要）
    d2 = savgol_filter(X_raw, window_length=21, polyorder=3, deriv=2, axis=1)
    ratio = (X_raw[:, idx_1940] / (X_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
    std = np.std(X_raw, axis=1, keepdims=True)
    raw_mean = np.mean(X_raw, axis=1, keepdims=True)  # 密度proxy
    return d2, ratio, std, raw_mean


make_features = make_features_A if not USE_CONFIG_B else make_features_B


# ============================================================
# 4. Multi-Seed Ensemble CV
# ============================================================
gkf = GroupKFold(n_splits=5)
X_test_raw = test[spec_cols].values

all_seed_test = []
all_seed_oof = []
all_seed_metrics = []

for seed_idx, base_seed in enumerate(ENSEMBLE_SEEDS):
    print(f"\n{'='*60}")
    is_original = "(= 元11.80の再現)" if base_seed == 42 else ""
    print(f"🌱 Seed {seed_idx+1}/{len(ENSEMBLE_SEEDS)} "
          f"(base={base_seed}) {is_original}")
    print(f"{'='*60}")

    final_lgb = np.zeros(len(test))
    oof_lgb = np.zeros(len(train))
    fold_rmses = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        train[spec_cols].values, y_train_log, groups
    )):
        va_species = train.iloc[va_idx]['樹種'].unique()
        tr_sp_nums = groups.iloc[tr_idx].values

        X_tr_raw = train[spec_cols].values[tr_idx]
        y_tr = y_train_log.iloc[tr_idx].values
        X_va_raw = train[spec_cols].values[va_idx]
        y_va = y_train_log.iloc[va_idx].values

        # ── Mixup（元と同じ設定）──
        fold_seed = base_seed + fold
        rng = np.random.RandomState(fold_seed)

        X_mix, y_mix = mixup_2way(
            X_tr_raw, y_tr, tr_sp_nums,
            MIXUP_N, MIXUP_ALPHA, rng
        )

        n_orig = len(X_tr_raw)
        X_tr_aug = np.vstack([X_tr_raw, X_mix])
        y_tr_aug = np.concatenate([y_tr, y_mix])

        if fold == 0 and seed_idx == 0:
            print(f"  元: {n_orig}, mixup: {len(X_mix)}, "
                  f"合計: {len(X_tr_aug)}")

        # ── 特徴量（A/B分岐）──
        if not USE_CONFIG_B:
            # Config A: SNV + d1
            snv_tr, d1_tr, ratio_tr, std_tr = make_features(X_tr_aug)
            snv_va, d1_va, ratio_va, std_va = make_features(X_va_raw)
            snv_te, d1_te, ratio_te, std_te = make_features(X_test_raw)

            # PCA（元データのみでfit）
            snv_tr_orig = apply_snv(X_tr_raw)
            pca = PCA(n_components=10, random_state=42)
            pca.fit(snv_tr_orig)
            pca_tr = pca.transform(snv_tr)
            pca_va = pca.transform(snv_va)
            pca_te = pca.transform(snv_te)

            # KNN（元データのみでfit）
            pca_tr_orig = pca.transform(snv_tr_orig)
            knn = NearestNeighbors(n_neighbors=5, metric='cosine')
            knn.fit(pca_tr_orig)

            _, ind_tr = knn.kneighbors(pca_tr, n_neighbors=6)
            knn_ymean_tr = np.zeros(len(X_tr_aug))
            for i in range(len(X_tr_aug)):
                nb = ind_tr[i]
                if i < n_orig:
                    valid = nb[nb != i][:5]
                else:
                    valid = nb[:5]
                knn_ymean_tr[i] = np.mean(y_tr[valid])
            knn_ymean_tr = knn_ymean_tr.reshape(-1, 1)

            _, ind_va = knn.kneighbors(pca_va, n_neighbors=5)
            knn_ymean_va = np.mean(y_tr[ind_va], axis=1).reshape(-1, 1)
            _, ind_te = knn.kneighbors(pca_te, n_neighbors=5)
            knn_ymean_te = np.mean(y_tr[ind_te], axis=1).reshape(-1, 1)

            feat_tr = np.hstack([snv_tr, d1_tr, pca_tr,
                                 knn_ymean_tr, ratio_tr, std_tr])
            feat_va = np.hstack([snv_va, d1_va, pca_va,
                                 knn_ymean_va, ratio_va, std_va])
            feat_te = np.hstack([snv_te, d1_te, pca_te,
                                 knn_ymean_te, ratio_te, std_te])

        else:
            # Config B: d2一本化
            d2_tr, ratio_tr, std_tr, rmean_tr = make_features(X_tr_aug)
            d2_va, ratio_va, std_va, rmean_va = make_features(X_va_raw)
            d2_te, ratio_te, std_te, rmean_te = make_features(X_test_raw)

            # PCA on d2（元データのみでfit）
            d2_tr_orig = savgol_filter(
                X_tr_raw, window_length=21, polyorder=3, deriv=2, axis=1)
            pca = PCA(n_components=10, random_state=42)
            pca.fit(d2_tr_orig)
            pca_tr = pca.transform(d2_tr)
            pca_va = pca.transform(d2_va)
            pca_te = pca.transform(d2_te)

            # KNN on d2 PCA
            pca_tr_orig = pca.transform(d2_tr_orig)
            knn = NearestNeighbors(n_neighbors=5, metric='cosine')
            knn.fit(pca_tr_orig)

            _, ind_tr = knn.kneighbors(pca_tr, n_neighbors=6)
            knn_ymean_tr = np.zeros(len(X_tr_aug))
            for i in range(len(X_tr_aug)):
                nb = ind_tr[i]
                if i < n_orig:
                    valid = nb[nb != i][:5]
                else:
                    valid = nb[:5]
                knn_ymean_tr[i] = np.mean(y_tr[valid])
            knn_ymean_tr = knn_ymean_tr.reshape(-1, 1)

            _, ind_va = knn.kneighbors(pca_va, n_neighbors=5)
            knn_ymean_va = np.mean(y_tr[ind_va], axis=1).reshape(-1, 1)
            _, ind_te = knn.kneighbors(pca_te, n_neighbors=5)
            knn_ymean_te = np.mean(y_tr[ind_te], axis=1).reshape(-1, 1)

            feat_tr = np.hstack([d2_tr, pca_tr,
                                 knn_ymean_tr, ratio_tr, std_tr, rmean_tr])
            feat_va = np.hstack([d2_va, pca_va,
                                 knn_ymean_va, ratio_va, std_va, rmean_va])
            feat_te = np.hstack([d2_te, pca_te,
                                 knn_ymean_te, ratio_te, std_te, rmean_te])

        if fold == 0 and seed_idx == 0:
            print(f"  📐 特徴量次元: {feat_tr.shape[1]}")

        # ── LightGBM（元と同一パラメータ）──
        lgb_model = lgb.LGBMRegressor(
            n_estimators=1000, learning_rate=0.03,
            max_depth=5, num_leaves=31,
            subsample=0.8, colsample_bytree=0.3,
            random_state=42, verbosity=-1
        )
        lgb_model.fit(
            feat_tr, y_tr_aug,
            eval_set=[(feat_va, y_va)],
            callbacks=[lgb.early_stopping(30, verbose=False)]
        )

        p_va = np.expm1(lgb_model.predict(feat_va))
        p_te = np.expm1(lgb_model.predict(feat_te))
        oof_lgb[va_idx] = p_va
        final_lgb += p_te / 5

        rmse = np.sqrt(mean_squared_error(np.expm1(y_va), p_va))
        fold_rmses.append(rmse)
        print(f"  Fold {fold+1} RMSE: {rmse:.4f}  "
              f"(valid: {list(va_species)})")

    oof_rmse = np.sqrt(mean_squared_error(np.expm1(y_train_log), oof_lgb))
    fold_mean = np.mean(fold_rmses)
    fold_std = np.std(fold_rmses)
    print(f"  🌟 Seed {seed_idx+1} OOF RMSE: {oof_rmse:.4f} "
          f"(Fold平均: {fold_mean:.4f} ± {fold_std:.4f})")

    all_seed_test.append(final_lgb)
    all_seed_oof.append(oof_lgb)
    all_seed_metrics.append({
        'seed': base_seed,
        'oof_rmse': oof_rmse,
        'fold_mean': fold_mean,
        'fold_std': fold_std
    })


# ============================================================
# 5. Ensemble集約
# ============================================================
final_ensemble = np.mean(all_seed_test, axis=0)
oof_ensemble = np.mean(all_seed_oof, axis=0)

y_true = np.expm1(y_train_log)
oof_rmse_ens = np.sqrt(mean_squared_error(y_true, oof_ensemble))

print(f"\n{'='*60}")
print(f"📊 Ensemble結果 ({len(ENSEMBLE_SEEDS)} seeds)")
print(f"{'='*60}")
print(f"  🌟 Ensemble OOF RMSE: {oof_rmse_ens:.4f}")

# 個別seedの結果一覧
print(f"\n  {'Seed':>6s} {'OOF RMSE':>10s} {'Fold平均':>10s} {'±':>8s}")
print(f"  {'─'*6} {'─'*10} {'─'*10} {'─'*8}")
for m in all_seed_metrics:
    marker = " ← 元11.80" if m['seed'] == 42 else ""
    print(f"  {m['seed']:>6d} {m['oof_rmse']:>10.4f} "
          f"{m['fold_mean']:>10.4f} {m['fold_std']:>8.4f}{marker}")

# seed間のばらつき
for i, pred in enumerate(all_seed_test):
    diff = np.sqrt(np.mean((pred - final_ensemble)**2))
    print(f"  Seed {ENSEMBLE_SEEDS[i]} vs Ensemble RMSD: {diff:.4f}")

# 樹種別
print(f"\n  {'樹種':12s} {'n':>4s} {'RMSE':>7s} {'bias':>7s}")
print(f"  {'─'*12} {'─'*4} {'─'*7} {'─'*7}")
for sp in sorted(train['樹種'].unique()):
    mask = train['樹種'] == sp
    y_s = train.loc[mask, '含水率'].values
    p_s = oof_ensemble[mask.values]
    r = np.sqrt(np.mean((y_s - p_s)**2))
    b = np.mean(y_s - p_s)
    print(f"  {sp:12s} {len(y_s):4d} {r:7.2f} {b:+7.2f}")


# ============================================================
# 6. 提出ファイル
# ============================================================
final_out = np.clip(final_ensemble, 0, None)
submit[1] = final_out
seeds_str = "_".join(str(s) for s in ENSEMBLE_SEEDS)
fname = f'submission_{config_name}_seeds{seeds_str}.csv'
submit.to_csv(fname, index=False, header=False)

print(f"\n✅ 提出ファイル: {fname}")
print(f"📈 予測分布: min={final_out.min():.1f}%, "
      f"median={np.median(final_out):.1f}%, "
      f"max={final_out.max():.1f}%")


# ============================================================
# 7. seed=42単独の提出も保存（11.80再現確認用）
# ============================================================
if 42 in ENSEMBLE_SEEDS:
    idx42 = ENSEMBLE_SEEDS.index(42)
    pred_42 = np.clip(all_seed_test[idx42], 0, None)
    submit_42 = submit.copy()
    submit_42[1] = pred_42
    fname_42 = f'submission_{config_name}_seed42_only.csv'
    submit_42.to_csv(fname_42, index=False, header=False)
    print(f"✅ seed42単独: {fname_42}")


# ============================================================
# 8. 全スコア比較
# ============================================================
print(f"\n{'='*60}")
print("📌 全スコア比較（過去 → 今回）")
print(f"{'='*60}")
print(f"  {'手法':<34s} {'OOF RMSE':>10s} {'Fold平均':>10s} {'LB RMSE':>10s}")
print(f"  {'─'*34} {'─'*10} {'─'*10} {'─'*10}")

for name, oof, fold_avg, lb in HISTORY:
    oof_str = f"{oof:.2f}" if oof is not None else "---"
    fold_str = f"{fold_avg:.2f}" if fold_avg is not None else "---"
    lb_str = f"{lb:.3f}" if lb is not None else "---"
    if lb is not None and lb == 11.80:
        marker = " ★BEST"
    elif lb is not None and lb == 12.249:
        marker = " ↓悪化"
    else:
        marker = ""
    print(f"  {name:<34s} {oof_str:>10s} {fold_str:>10s} "
          f"{lb_str:>10s}{marker}")

# 今回
today_name = f"{config_name} (ensemble)"
print(f"  {today_name:<34s} {oof_rmse_ens:>10.2f} {'---':>10s} "
      f"{'???':>10s} ← 今回(ensemble)")

# seed42単独も記録
if 42 in ENSEMBLE_SEEDS:
    m42 = all_seed_metrics[ENSEMBLE_SEEDS.index(42)]
    today_42 = f"{config_name} (seed42のみ)"
    print(f"  {today_42:<34s} {m42['oof_rmse']:>10.2f} "
          f"{m42['fold_mean']:>10.2f} {'???':>10s} ← 今回(seed42)")


# ============================================================
# 9. 判断ガイド
# ============================================================
print(f"\n{'='*60}")
print("🧭 提出判断ガイド")
print(f"{'='*60}")
print(f"""
  ■ 提出候補が2ファイルあります:
  
  1. {fname}
     → {len(ENSEMBLE_SEEDS)}seed平均。分散低減による安定向上を狙う。
     
  2. {fname_42 if 42 in ENSEMBLE_SEEDS else '(なし)'}
     → seed42単独。11.80の完全再現を確認。
  
  ■ 推奨提出順:
     まず seed42単独 → 11.80が再現されるか確認
     次に ensemble  → 改善するか確認
  
  ■ LB結果ごとの次手:
  
  seed42 = 11.80再現 & ensemble < 11.80:
    → multi-seedが効いた！seed数を5に増やす
  
  seed42 = 11.80再現 & ensemble > 11.80:
    → 他seedが足を引っ張っている。seed42単独がBEST
    → 次はConfig B (d2一本化) を試す
  
  seed42 ≠ 11.80 (再現せず):
    → 元コードと微妙な差異あり。差分を確認
""")

print(f"\n{'='*60}")
print("📊 Config B（d2一本化）を試す場合:")
print(f"{'='*60}")
print(f"""
  USE_CONFIG_B = True に変更して再実行
  
  期待される効果:
  - SNV+d1の2×1568次元 → d2の1×1568次元（次元半減）
  - ベースライン変動を微分で完全除去
  - 樹種固有の散乱パターンが消え、水ピーク形状のみ残る
  - 過学習リスク低下 → 未知樹種への汎化向上の可能性
  
  リスク:
  - d2はノイズを増幅する → SN比低下
  - SNVが持っていた密度情報が完全消失
    → raw_mean特徴量で補完済み
""")

🧪 Config: A_safe_multiseed
   Mixup: 500 samples, α=0.3
   Seeds: [42, 100, 200]
   d2 only: False

🌱 Seed 1/3 (base=42) (= 元11.80の再現)
  元: 940, mixup: 500, 合計: 1440
  📐 特徴量次元: 3123
  Fold 1 RMSE: 7.9258  (valid: ['ウエンジ', 'トチ'])
  Fold 2 RMSE: 17.8934  (valid: ['チェリー', 'ヒノキ'])
  Fold 3 RMSE: 20.7025  (valid: ['ウォールナット', 'クリ'])
  Fold 4 RMSE: 19.7955  (valid: ['ナラ', 'ベイマツ', 'ホワイトオーク'])
  Fold 5 RMSE: 21.4266  (valid: ['イチョウ', 'スプルース', '米ヒバ'])
  🌟 Seed 1 OOF RMSE: 18.0379 (Fold平均: 17.5488 ± 4.9548)

🌱 Seed 2/3 (base=100) 
  Fold 1 RMSE: 6.8331  (valid: ['ウエンジ', 'トチ'])
  Fold 2 RMSE: 18.2744  (valid: ['チェリー', 'ヒノキ'])
  Fold 3 RMSE: 19.1797  (valid: ['ウォールナット', 'クリ'])
  Fold 4 RMSE: 19.6287  (valid: ['ナラ', 'ベイマツ', 'ホワイトオーク'])
  Fold 5 RMSE: 20.7792  (valid: ['イチョウ', 'スプルース', '米ヒバ'])
  🌟 Seed 2 OOF RMSE: 17.5235 (Fold平均: 16.9390 ± 5.1169)

🌱 Seed 3/3 (base=200) 
  Fold 1 RMSE: 7.9149  (valid: ['ウエンジ', 'トチ'])
  Fold 2 RMSE: 18.6628  (valid: ['チェリー', 'ヒノキ'])
  Fold 3 RMSE: 20.2949  (valid: ['

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 過去スコア記録
# ============================================================
HISTORY = [
    ("LGB単独(元特徴量)",         17.21, None,  12.615),
    ("元Blend(LGB/PLS/Ridge)",   14.10, None,  12.647),
    ("正則化強化(LGB単独)",       17.60, None,  12.760),
    ("Huber Loss",               17.53, None,  12.770),
    ("PLS予測を特徴量追加",       15.65, None,  12.800),
    ("逆距離加重KNN",            17.13, None,  12.940),
    ("d2(二次微分)追加",          16.12, None,  13.410),
    ("物理特徴量53個追加",        12.68, None,  14.500),
    ("Mixup(500,α=0.3)seed42",  None,  None,  11.80),
    ("MultiSeed5+2w2000+3w1000", None,  None,  12.249),
    ("SafeMultiSeed3(ensemble)", 17.64, 17.31, 11.870),
]

# ============================================================
# 1. データ読み込み
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit_template = pd.read_csv('data/sample_submit.csv', header=None)

train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)
idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))

X_train_raw = train[spec_cols].values
X_test_raw  = test[spec_cols].values


def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s


# ============================================================
# 2. EDA: Train vs Test 分析
# ============================================================
print("=" * 60)
print("📊 EDA: Train vs Test 分析")
print("=" * 60)

# --- 2a. 基本統計 ---
print("\n--- 2a. 基本情報 ---")
train_species = sorted(train['樹種'].unique())
test_species  = sorted(test['樹種'].unique())
print(f"  Train: {len(train)} samples, {len(train_species)} species")
print(f"  Test:  {len(test)} samples, {len(test_species)} species")
print(f"  Train樹種: {train_species}")
print(f"  Test樹種:  {test_species}")
print(f"  含水率: min={train['含水率'].min():.1f}, "
      f"median={train['含水率'].median():.1f}, "
      f"max={train['含水率'].max():.1f}")

# --- 2b. PCA空間でのTrain vs Test ---
print("\n--- 2b. PCA空間でのTrain vs Test位置関係 ---")
snv_all = apply_snv(np.vstack([X_train_raw, X_test_raw]))
pca_eda = PCA(n_components=10, random_state=42)
pca_eda.fit(apply_snv(X_train_raw))  # trainのみでfit
pca_train_eda = pca_eda.transform(apply_snv(X_train_raw))
pca_test_eda  = pca_eda.transform(apply_snv(X_test_raw))

print(f"  PCA累積寄与率: {pca_eda.explained_variance_ratio_.cumsum()[:5]}")

# 樹種ごとのPCA重心
print(f"\n  {'樹種':14s} {'PC1':>7s} {'PC2':>7s} {'PC3':>7s} {'n':>5s} {'set':>5s}")
print(f"  {'─'*14} {'─'*7} {'─'*7} {'─'*7} {'─'*5} {'─'*5}")

centroids_train = {}
for sp in train_species:
    mask = train['樹種'] == sp
    c = pca_train_eda[mask.values].mean(axis=0)
    centroids_train[sp] = c
    n = mask.sum()
    print(f"  {sp:14s} {c[0]:+7.3f} {c[1]:+7.3f} {c[2]:+7.3f} {n:5d} train")

centroids_test = {}
for sp in test_species:
    mask = test['樹種'] == sp
    c = pca_test_eda[mask.values].mean(axis=0)
    centroids_test[sp] = c
    n = mask.sum()
    print(f"  {sp:14s} {c[0]:+7.3f} {c[1]:+7.3f} {c[2]:+7.3f} {n:5d} TEST")

# 各Test樹種に最も近いTrain樹種（ユークリッド距離）
print(f"\n  --- 各Test樹種の最近傍Train樹種 ---")
print(f"  {'Test樹種':12s} → {'1st':12s} {'dist':>6s}  {'2nd':12s} {'dist':>6s}")
for tsp, tc in centroids_test.items():
    dists = {}
    for trsp, trc in centroids_train.items():
        d = np.sqrt(np.sum((tc[:5] - trc[:5])**2))
        dists[trsp] = d
    sorted_d = sorted(dists.items(), key=lambda x: x[1])
    print(f"  {tsp:12s} → {sorted_d[0][0]:12s} {sorted_d[0][1]:6.3f}  "
          f"{sorted_d[1][0]:12s} {sorted_d[1][1]:6.3f}")


# --- 2c. 樹種不変特徴量の同定 ---
print(f"\n--- 2c. 樹種不変特徴量の同定 ---")
print("  各波数における含水率との相関を樹種別に計算...")

snv_train = apply_snv(X_train_raw)
d1_train = savgol_filter(snv_train, window_length=15, polyorder=2, deriv=1, axis=1)
y_raw = train['含水率'].values

# SNV特徴量について樹種別相関を計算
n_feats = snv_train.shape[1]
min_abs_corrs = np.zeros(n_feats)
mean_corrs    = np.zeros(n_feats)
sign_consistent = np.zeros(n_feats, dtype=bool)

for j in range(n_feats):
    corrs = []
    for sp in train_species:
        mask = (train['樹種'] == sp).values
        if mask.sum() < 10:
            continue
        r = np.corrcoef(snv_train[mask, j], y_raw[mask])[0, 1]
        if np.isnan(r):
            continue
        corrs.append(r)
    if len(corrs) >= 5:
        min_abs_corrs[j] = min(abs(c) for c in corrs)
        mean_corrs[j]    = np.mean(corrs)
        signs = [c > 0 for c in corrs]
        sign_consistent[j] = all(signs) or not any(signs)

# 上位20の樹種不変特徴量
top_invariant_idx = np.argsort(-min_abs_corrs)[:20]
print(f"\n  樹種不変度TOP20 (SNV空間):")
print(f"  {'Rank':>4s} {'波数(cm-1)':>12s} {'波長(nm)':>10s} "
      f"{'min|r|':>8s} {'mean_r':>8s} {'符号一致':>8s}")
for rank, idx in enumerate(top_invariant_idx):
    wn = wavenumbers[idx]
    wl = wavelengths[idx]
    print(f"  {rank+1:4d} {wn:12.1f} {wl:10.0f} "
          f"{min_abs_corrs[idx]:8.3f} {mean_corrs[idx]:+8.3f} "
          f"{'✓' if sign_consistent[idx] else '✗':>8s}")

# 樹種不変特徴量のマスク作成
INVARIANT_THRESHOLD = 0.3
invariant_mask = (min_abs_corrs >= INVARIANT_THRESHOLD) & sign_consistent
n_invariant = invariant_mask.sum()
print(f"\n  閾値 min|r| >= {INVARIANT_THRESHOLD} & 符号一致: {n_invariant} / {n_feats} 特徴量")

# d1についても同様
min_abs_corrs_d1 = np.zeros(d1_train.shape[1])
sign_consistent_d1 = np.zeros(d1_train.shape[1], dtype=bool)
for j in range(d1_train.shape[1]):
    corrs = []
    for sp in train_species:
        mask = (train['樹種'] == sp).values
        if mask.sum() < 10:
            continue
        r = np.corrcoef(d1_train[mask, j], y_raw[mask])[0, 1]
        if np.isnan(r):
            continue
        corrs.append(r)
    if len(corrs) >= 5:
        min_abs_corrs_d1[j] = min(abs(c) for c in corrs)
        signs = [c > 0 for c in corrs]
        sign_consistent_d1[j] = all(signs) or not any(signs)

invariant_mask_d1 = (min_abs_corrs_d1 >= INVARIANT_THRESHOLD) & sign_consistent_d1
n_inv_d1 = invariant_mask_d1.sum()
print(f"  d1空間: {n_inv_d1} / {d1_train.shape[1]} 特徴量が閾値通過")

# 水バンド周辺の確認
print(f"\n  --- 水バンド周辺の樹種不変度 ---")
water_bands = [
    ("O-H 1st overtone", 6800, 7100),
    ("O-H combination",  5000, 5350),
    ("Cellulose O-H",    4500, 4800),
]
for name, wn_lo, wn_hi in water_bands:
    band_mask = (wavenumbers >= wn_lo) & (wavenumbers <= wn_hi)
    n_band = band_mask.sum()
    n_inv_band = (band_mask & invariant_mask).sum()
    avg_minr = min_abs_corrs[band_mask].mean()
    print(f"  {name:20s} ({wn_lo}-{wn_hi} cm-1): "
          f"{n_inv_band}/{n_band} invariant, avg_min|r|={avg_minr:.3f}")


# ============================================================
# 3. 実験用共通関数
# ============================================================
def mixup_augmentation(X, y, species, n_augment=500, alpha=0.3, seed=42):
    rng = np.random.RandomState(seed)
    unique_species = np.unique(species)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        sp1, sp2 = rng.choice(unique_species, size=2, replace=False)
        idx1 = rng.choice(np.where(species == sp1)[0])
        idx2 = rng.choice(np.where(species == sp2)[0])
        lam = rng.beta(alpha, alpha)
        X_aug.append(lam * X[idx1] + (1 - lam) * X[idx2])
        y_aug.append(lam * y[idx1] + (1 - lam) * y[idx2])
    return np.array(X_aug), np.array(y_aug)


def run_experiment(name, alpha, feat_func, lgb_params_override=None):
    """
    1つの実験を実行してOOF/テスト予測を返す
    feat_func: (X_tr_aug, X_va, X_te, X_tr_orig, y_tr) → (feat_tr, feat_va, feat_te)
    """
    print(f"\n{'='*60}")
    print(f"🔬 実験: {name}")
    print(f"{'='*60}")

    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred   = np.zeros(len(train))
    fold_rmses = []

    default_params = dict(
        n_estimators=1000, learning_rate=0.03,
        max_depth=5, num_leaves=31,
        subsample=0.8, colsample_bytree=0.3,
        random_state=42, verbosity=-1
    )
    if lgb_params_override:
        default_params.update(lgb_params_override)

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups
    )):
        va_species = train.iloc[va_idx]['樹種'].unique()
        tr_sp_nums = groups.iloc[tr_idx].values

        X_tr_raw_f = X_train_raw[tr_idx]
        y_tr       = y_train_log.iloc[tr_idx].values
        X_va_raw_f = X_train_raw[va_idx]
        y_va       = y_train_log.iloc[va_idx].values

        # Mixup
        X_mix, y_mix = mixup_augmentation(
            X_tr_raw_f, y_tr, tr_sp_nums,
            n_augment=500, alpha=alpha, seed=42 + fold
        )
        n_orig = len(X_tr_raw_f)
        X_tr_aug = np.vstack([X_tr_raw_f, X_mix])
        y_tr_aug = np.concatenate([y_tr, y_mix])

        # 特徴量作成（実験ごとに異なる関数）
        feat_tr, feat_va, feat_te = feat_func(
            X_tr_aug, X_va_raw_f, X_test_raw, X_tr_raw_f, y_tr
        )

        if fold == 0:
            print(f"  データ: orig={n_orig}, aug={len(X_tr_aug)}, "
                  f"feat_dim={feat_tr.shape[1]}")

        # LightGBM
        model = lgb.LGBMRegressor(**default_params)
        model.fit(
            feat_tr, y_tr_aug,
            eval_set=[(feat_va, y_va)],
            callbacks=[lgb.early_stopping(30, verbose=False)]
        )

        p_va = np.expm1(model.predict(feat_va))
        p_te = np.expm1(model.predict(feat_te))
        oof_pred[va_idx] = p_va
        final_pred += p_te / 5

        rmse = np.sqrt(mean_squared_error(np.expm1(y_va), p_va))
        fold_rmses.append(rmse)
        print(f"  Fold {fold+1} RMSE: {rmse:.4f}  "
              f"(valid: {list(va_species)})")

    y_true = np.expm1(y_train_log)
    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    fold_mean = np.mean(fold_rmses)
    fold_std  = np.std(fold_rmses)
    print(f"\n  🌟 OOF RMSE: {oof_rmse:.4f}")
    print(f"  📊 Fold平均: {fold_mean:.4f} ± {fold_std:.4f}")

    # 提出ファイル
    out = submit_template.copy()
    out[1] = np.clip(final_pred, 0, None)
    safe_name = name.replace(" ", "_").replace("(", "").replace(")", "")
    fname = f'submission_{safe_name}.csv'
    out.to_csv(fname, index=False, header=False)
    print(f"  ✅ {fname}")

    return {
        'name': name, 'oof_rmse': oof_rmse,
        'fold_mean': fold_mean, 'fold_std': fold_std,
        'test_pred': final_pred, 'oof_pred': oof_pred,
        'fname': fname,
    }


# ============================================================
# 4. 特徴量関数の定義
# ============================================================

def feat_baseline(X_tr_aug, X_va, X_te, X_tr_orig, y_tr):
    """11.80再現用：SNV + d1 + PCA + KNN"""
    n_orig = len(X_tr_orig)

    snv_tr = apply_snv(X_tr_aug)
    d1_tr  = savgol_filter(snv_tr, 15, 2, deriv=1, axis=1)
    snv_va = apply_snv(X_va)
    d1_va  = savgol_filter(snv_va, 15, 2, deriv=1, axis=1)
    snv_te = apply_snv(X_te)
    d1_te  = savgol_filter(snv_te, 15, 2, deriv=1, axis=1)

    ratio_tr = (X_tr_aug[:, idx_1940] / (X_tr_aug[:, idx_1300]+1e-8)).reshape(-1,1)
    ratio_va = (X_va[:, idx_1940] / (X_va[:, idx_1300]+1e-8)).reshape(-1,1)
    ratio_te = (X_te[:, idx_1940] / (X_te[:, idx_1300]+1e-8)).reshape(-1,1)
    std_tr = np.std(X_tr_aug, axis=1, keepdims=True)
    std_va = np.std(X_va, axis=1, keepdims=True)
    std_te = np.std(X_te, axis=1, keepdims=True)

    pca = PCA(n_components=10, random_state=42)
    pca.fit(apply_snv(X_tr_orig))
    pca_tr = pca.transform(snv_tr)
    pca_va = pca.transform(snv_va)
    pca_te = pca.transform(snv_te)

    pca_orig = pca.transform(apply_snv(X_tr_orig))
    knn = NearestNeighbors(n_neighbors=5, metric='cosine')
    knn.fit(pca_orig)

    _, ind_tr = knn.kneighbors(pca_tr, n_neighbors=6)
    knn_tr = np.zeros(len(X_tr_aug))
    for i in range(len(X_tr_aug)):
        nb = ind_tr[i]
        valid = nb[nb != i][:5] if i < n_orig else nb[:5]
        knn_tr[i] = np.mean(y_tr[:n_orig][valid])
    knn_tr = knn_tr.reshape(-1, 1)

    _, ind_va = knn.kneighbors(pca_va, 5)
    knn_va = np.mean(y_tr[ind_va], axis=1).reshape(-1, 1)
    _, ind_te = knn.kneighbors(pca_te, 5)
    knn_te = np.mean(y_tr[ind_te], axis=1).reshape(-1, 1)

    return (np.hstack([snv_tr, d1_tr, pca_tr, knn_tr, ratio_tr, std_tr]),
            np.hstack([snv_va, d1_va, pca_va, knn_va, ratio_va, std_va]),
            np.hstack([snv_te, d1_te, pca_te, knn_te, ratio_te, std_te]))


def feat_invariant_only(X_tr_aug, X_va, X_te, X_tr_orig, y_tr):
    """樹種不変特徴量のみ使用"""
    n_orig = len(X_tr_orig)

    snv_tr = apply_snv(X_tr_aug)[:, invariant_mask]
    d1_all = savgol_filter(apply_snv(X_tr_aug), 15, 2, deriv=1, axis=1)
    d1_tr  = d1_all[:, invariant_mask_d1]
    snv_va = apply_snv(X_va)[:, invariant_mask]
    d1_va  = savgol_filter(apply_snv(X_va), 15, 2, deriv=1, axis=1)[:, invariant_mask_d1]
    snv_te = apply_snv(X_te)[:, invariant_mask]
    d1_te  = savgol_filter(apply_snv(X_te), 15, 2, deriv=1, axis=1)[:, invariant_mask_d1]

    ratio_tr = (X_tr_aug[:, idx_1940] / (X_tr_aug[:, idx_1300]+1e-8)).reshape(-1,1)
    ratio_va = (X_va[:, idx_1940] / (X_va[:, idx_1300]+1e-8)).reshape(-1,1)
    ratio_te = (X_te[:, idx_1940] / (X_te[:, idx_1300]+1e-8)).reshape(-1,1)
    std_tr = np.std(X_tr_aug, axis=1, keepdims=True)
    std_va = np.std(X_va, axis=1, keepdims=True)
    std_te = np.std(X_te, axis=1, keepdims=True)

    # PCA on invariant features
    pca = PCA(n_components=min(10, n_invariant), random_state=42)
    pca.fit(apply_snv(X_tr_orig)[:, invariant_mask])
    pca_tr = pca.transform(snv_tr)
    pca_va = pca.transform(snv_va)
    pca_te = pca.transform(snv_te)

    pca_orig = pca.transform(apply_snv(X_tr_orig)[:, invariant_mask])
    knn = NearestNeighbors(n_neighbors=5, metric='cosine')
    knn.fit(pca_orig)

    _, ind_tr = knn.kneighbors(pca_tr, n_neighbors=6)
    knn_tr = np.zeros(len(X_tr_aug))
    for i in range(len(X_tr_aug)):
        nb = ind_tr[i]
        valid = nb[nb != i][:5] if i < n_orig else nb[:5]
        knn_tr[i] = np.mean(y_tr[:n_orig][valid])
    knn_tr = knn_tr.reshape(-1, 1)

    _, ind_va = knn.kneighbors(pca_va, 5)
    knn_va = np.mean(y_tr[ind_va], axis=1).reshape(-1, 1)
    _, ind_te = knn.kneighbors(pca_te, 5)
    knn_te = np.mean(y_tr[ind_te], axis=1).reshape(-1, 1)

    return (np.hstack([snv_tr, d1_tr, pca_tr, knn_tr, ratio_tr, std_tr]),
            np.hstack([snv_va, d1_va, pca_va, knn_va, ratio_va, std_va]),
            np.hstack([snv_te, d1_te, pca_te, knn_te, ratio_te, std_te]))


def feat_water_bands_only(X_tr_aug, X_va, X_te, X_tr_orig, y_tr):
    """水バンド周辺のみ使用（樹種成分を最小化）"""
    n_orig = len(X_tr_orig)

    # 水バンドのマスク: 6700-7200, 4900-5400 cm-1
    water_mask = ((wavenumbers >= 6700) & (wavenumbers <= 7200)) | \
                 ((wavenumbers >= 4900) & (wavenumbers <= 5400))

    snv_tr = apply_snv(X_tr_aug)[:, water_mask]
    snv_va = apply_snv(X_va)[:, water_mask]
    snv_te = apply_snv(X_te)[:, water_mask]

    d1_tr = savgol_filter(apply_snv(X_tr_aug), 15, 2, deriv=1, axis=1)[:, water_mask]
    d1_va = savgol_filter(apply_snv(X_va), 15, 2, deriv=1, axis=1)[:, water_mask]
    d1_te = savgol_filter(apply_snv(X_te), 15, 2, deriv=1, axis=1)[:, water_mask]

    ratio_tr = (X_tr_aug[:, idx_1940] / (X_tr_aug[:, idx_1300]+1e-8)).reshape(-1,1)
    ratio_va = (X_va[:, idx_1940] / (X_va[:, idx_1300]+1e-8)).reshape(-1,1)
    ratio_te = (X_te[:, idx_1940] / (X_te[:, idx_1300]+1e-8)).reshape(-1,1)
    std_tr = np.std(X_tr_aug, axis=1, keepdims=True)
    std_va = np.std(X_va, axis=1, keepdims=True)
    std_te = np.std(X_te, axis=1, keepdims=True)

    n_water = water_mask.sum()
    pca = PCA(n_components=min(10, n_water), random_state=42)
    pca.fit(apply_snv(X_tr_orig)[:, water_mask])
    pca_tr = pca.transform(snv_tr)
    pca_va = pca.transform(snv_va)
    pca_te = pca.transform(snv_te)

    pca_orig = pca.transform(apply_snv(X_tr_orig)[:, water_mask])
    knn = NearestNeighbors(n_neighbors=5, metric='cosine')
    knn.fit(pca_orig)

    _, ind_tr = knn.kneighbors(pca_tr, n_neighbors=6)
    knn_tr = np.zeros(len(X_tr_aug))
    for i in range(len(X_tr_aug)):
        nb = ind_tr[i]
        valid = nb[nb != i][:5] if i < n_orig else nb[:5]
        knn_tr[i] = np.mean(y_tr[:n_orig][valid])
    knn_tr = knn_tr.reshape(-1, 1)

    _, ind_va = knn.kneighbors(pca_va, 5)
    knn_va = np.mean(y_tr[ind_va], axis=1).reshape(-1, 1)
    _, ind_te = knn.kneighbors(pca_te, 5)
    knn_te = np.mean(y_tr[ind_te], axis=1).reshape(-1, 1)

    return (np.hstack([snv_tr, d1_tr, pca_tr, knn_tr, ratio_tr, std_tr]),
            np.hstack([snv_va, d1_va, pca_va, knn_va, ratio_va, std_va]),
            np.hstack([snv_te, d1_te, pca_te, knn_te, ratio_te, std_te]))


def feat_d2_compact(X_tr_aug, X_va, X_te, X_tr_orig, y_tr):
    """d2一本化（SNV+d1を排除→次元半減、ベースライン完全除去）"""
    n_orig = len(X_tr_orig)

    d2_tr = savgol_filter(X_tr_aug, 21, 3, deriv=2, axis=1)
    d2_va = savgol_filter(X_va, 21, 3, deriv=2, axis=1)
    d2_te = savgol_filter(X_te, 21, 3, deriv=2, axis=1)

    ratio_tr = (X_tr_aug[:, idx_1940] / (X_tr_aug[:, idx_1300]+1e-8)).reshape(-1,1)
    ratio_va = (X_va[:, idx_1940] / (X_va[:, idx_1300]+1e-8)).reshape(-1,1)
    ratio_te = (X_te[:, idx_1940] / (X_te[:, idx_1300]+1e-8)).reshape(-1,1)
    std_tr = np.std(X_tr_aug, axis=1, keepdims=True)
    std_va = np.std(X_va, axis=1, keepdims=True)
    std_te = np.std(X_te, axis=1, keepdims=True)
    mean_tr = np.mean(X_tr_aug, axis=1, keepdims=True)
    mean_va = np.mean(X_va, axis=1, keepdims=True)
    mean_te = np.mean(X_te, axis=1, keepdims=True)

    d2_orig = savgol_filter(X_tr_orig, 21, 3, deriv=2, axis=1)
    pca = PCA(n_components=10, random_state=42)
    pca.fit(d2_orig)
    pca_tr = pca.transform(d2_tr)
    pca_va = pca.transform(d2_va)
    pca_te = pca.transform(d2_te)

    pca_orig = pca.transform(d2_orig)
    knn = NearestNeighbors(n_neighbors=5, metric='cosine')
    knn.fit(pca_orig)

    _, ind_tr = knn.kneighbors(pca_tr, n_neighbors=6)
    knn_tr = np.zeros(len(X_tr_aug))
    for i in range(len(X_tr_aug)):
        nb = ind_tr[i]
        valid = nb[nb != i][:5] if i < n_orig else nb[:5]
        knn_tr[i] = np.mean(y_tr[:n_orig][valid])
    knn_tr = knn_tr.reshape(-1, 1)

    _, ind_va = knn.kneighbors(pca_va, 5)
    knn_va = np.mean(y_tr[ind_va], axis=1).reshape(-1, 1)
    _, ind_te = knn.kneighbors(pca_te, 5)
    knn_te = np.mean(y_tr[ind_te], axis=1).reshape(-1, 1)

    return (np.hstack([d2_tr, pca_tr, knn_tr, ratio_tr, std_tr, mean_tr]),
            np.hstack([d2_va, pca_va, knn_va, ratio_va, std_va, mean_va]),
            np.hstack([d2_te, pca_te, knn_te, ratio_te, std_te, mean_te]))


# ============================================================
# 5. 全実験を実行
# ============================================================
experiments = [
    ("A_baseline_a03",       0.3, feat_baseline),
    ("B_alpha05",            0.5, feat_baseline),
    ("C_alpha10",            1.0, feat_baseline),
    ("D_invariant_only",     0.3, feat_invariant_only),
    ("E_water_bands_only",   0.3, feat_water_bands_only),
    ("F_d2_compact",         0.3, feat_d2_compact),
]

all_results = []

for exp_name, alpha, feat_func in experiments:
    result = run_experiment(exp_name, alpha, feat_func)
    all_results.append(result)


# ============================================================
# 6. 全実験比較
# ============================================================
print(f"\n{'='*60}")
print("📊 今回の実験比較")
print(f"{'='*60}")
print(f"  {'実験名':<26s} {'OOF':>8s} {'Fold平均':>8s} {'±':>7s} {'次元':>6s}")
print(f"  {'─'*26} {'─'*8} {'─'*8} {'─'*7} {'─'*6}")
for r in all_results:
    print(f"  {r['name']:<26s} {r['oof_rmse']:>8.2f} "
          f"{r['fold_mean']:>8.2f} {r['fold_std']:>7.2f}")

# テスト予測の差分
base_pred = all_results[0]['test_pred']
print(f"\n  --- テスト予測のbaseline(A)との差分 ---")
for r in all_results[1:]:
    diff = r['test_pred'] - base_pred
    rmsd = np.sqrt(np.mean(diff**2))
    print(f"  {r['name']:<26s} RMSD={rmsd:.3f}, "
          f"mean={np.mean(diff):+.3f}, max|diff|={np.max(np.abs(diff)):.3f}")


# ============================================================
# 7. アンサンブル候補の探索
# ============================================================
print(f"\n{'='*60}")
print("📊 アンサンブル候補（2つ組み合わせ）")
print(f"{'='*60}")

from itertools import combinations

best_ens_oof = 999
best_ens_pair = None

y_true_real = np.expm1(y_train_log)

for (r1, r2) in combinations(all_results, 2):
    for w in [0.3, 0.5, 0.7]:
        oof_ens = w * r1['oof_pred'] + (1 - w) * r2['oof_pred']
        rmse_ens = np.sqrt(mean_squared_error(y_true_real, oof_ens))
        if rmse_ens < best_ens_oof:
            best_ens_oof = rmse_ens
            best_ens_pair = (r1['name'], r2['name'], w, rmse_ens)

print(f"  ⚠️ OOF最良のアンサンブル（LBとは逆相関の可能性あり）:")
print(f"  {best_ens_pair[0]} × {best_ens_pair[2]:.1f} + "
      f"{best_ens_pair[1]} × {1-best_ens_pair[2]:.1f} "
      f"→ OOF RMSE: {best_ens_pair[3]:.4f}")

# 最も異なる予測の組み合わせ（多様性重視）
print(f"\n  --- 予測の多様性（相関が低いペア＝アンサンブル効果高い）---")
for (r1, r2) in combinations(all_results, 2):
    corr = np.corrcoef(r1['test_pred'], r2['test_pred'])[0, 1]
    print(f"  {r1['name']:<22s} × {r2['name']:<22s} corr={corr:.4f}")


# ============================================================
# 8. 全スコア比較表（過去 + 今回）
# ============================================================
print(f"\n{'='*60}")
print("📌 全スコア比較（過去 → 今回）")
print(f"{'='*60}")
print(f"  {'手法':<36s} {'OOF':>8s} {'Fold平均':>8s} {'LB':>8s}")
print(f"  {'─'*36} {'─'*8} {'─'*8} {'─'*8}")

for name, oof, fold_avg, lb in HISTORY:
    oof_s  = f"{oof:.2f}" if oof is not None else "---"
    fold_s = f"{fold_avg:.2f}" if fold_avg is not None else "---"
    lb_s   = f"{lb:.3f}" if lb is not None else "---"
    if lb == 11.80:
        m = " ★BEST"
    elif lb is not None and lb > 12.0:
        m = " ↓"
    else:
        m = ""
    print(f"  {name:<36s} {oof_s:>8s} {fold_s:>8s} {lb_s:>8s}{m}")

print(f"  {'─'*36} {'─'*8} {'─'*8} {'─'*8}")
for r in all_results:
    label = f"今回: {r['name']}"
    print(f"  {label:<36s} {r['oof_rmse']:>8.2f} "
          f"{r['fold_mean']:>8.2f} {'???':>8s}")


# ============================================================
# 9. 提出判断ガイド
# ============================================================
print(f"\n{'='*60}")
print("🧭 提出判断ガイド")
print(f"{'='*60}")

# OOFが最も悪い実験（＝逆相関法則で最もLBが良い可能性）
worst_oof = max(all_results, key=lambda x: x['oof_rmse'])
best_oof  = min(all_results, key=lambda x: x['oof_rmse'])

print(f"""
  ■ 過去の法則（OOF↓→LB↑）に従う場合:
    最有力: {worst_oof['name']} (OOF={worst_oof['oof_rmse']:.2f})
    → ファイル: {worst_oof['fname']}

  ■ OOFを信じる場合:
    最有力: {best_oof['name']} (OOF={best_oof['oof_rmse']:.2f})
    → ファイル: {best_oof['fname']}

  ■ 安全策:
    A_baseline_a03 → 11.80の再現を確認
    → ファイル: {all_results[0]['fname']}

  ■ 推奨提出順（1日3回制限の場合）:
    1st: A_baseline_a03      → 11.80再現確認
    2nd: {worst_oof['name']:<22s} → 逆相関法則に賭ける
    3rd: {best_oof['name']:<22s} → OOF法則に賭ける

  ■ 全提出ファイル一覧:
""")

for r in all_results:
    print(f"    {r['fname']}")

print(f"""
  ■ 次回以降の検討事項:
    - EDAで判明した「各Test樹種に近いTrain樹種」情報の活用
    - 樹種不変特徴量の閾値調整 (現在: min|r|>={INVARIANT_THRESHOLD})
    - Mixup時に「Test樹種に近いTrain樹種ペア」を優先的に混合
    - 含水率の高低域で別モデルを構築（区分回帰）
""")

📊 EDA: Train vs Test 分析

--- 2a. 基本情報 ---
  Train: 1210 samples, 12 species
  Test:  550 samples, 6 species
  Train樹種: ['イチョウ', 'ウエンジ', 'ウォールナット', 'クリ', 'スプルース', 'チェリー', 'トチ', 'ナラ', 'ヒノキ', 'ベイマツ', 'ホワイトオーク', '米ヒバ']
  Test樹種:  ['クスノキ', 'ケヤキ', 'スギ', 'タモ', 'チーク', 'ヤマザクラ']
  含水率: min=0.8, median=28.1, max=216.1

--- 2b. PCA空間でのTrain vs Test位置関係 ---
  PCA累積寄与率: [0.62380134 0.88944498 0.96216633 0.9821446  0.98751215]

  樹種                 PC1     PC2     PC3     n   set
  ────────────── ─────── ─────── ─────── ───── ─────
  イチョウ            +0.480  +1.558  +0.157    94 train
  ウエンジ            -8.164  +3.604  -2.783    87 train
  ウォールナット         -2.789  +4.745  +1.287   110 train
  クリ              +1.421  +0.343  +0.248    91 train
  スプルース           +0.610  -2.080  -0.258    97 train
  チェリー            +2.860  +0.396  -0.382    88 train
  トチ              +0.002  -1.426  +0.954   183 train
  ナラ              +1.678  -0.332  -0.167   107 train
  ヒノキ             +1.647  -1.838  +0.112   141 train


In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 過去スコア記録
# ============================================================
HISTORY = [
    ("LGB単独(元特徴量)",          17.21, 12.615),
    ("元Blend(LGB/PLS/Ridge)",    14.10, 12.647),
    ("正則化強化(LGB単独)",        17.60, 12.760),
    ("Huber Loss",                17.53, 12.770),
    ("PLS予測を特徴量追加",        15.65, 12.800),
    ("逆距離加重KNN",             17.13, 12.940),
    ("d2(二次微分)追加",           16.12, 13.410),
    ("物理特徴量53個追加",         12.68, 14.500),
    ("Mixup(500,α=0.3)seed42",   18.04, 11.800),
    ("MultiSeed5+2w2000+3w1000", None,  12.249),
    ("SafeMultiSeed3(ensemble)", 17.64, 11.870),
    ("B_alpha05",                17.82, 12.326),
    ("E_water_bands_only",       19.43, 12.656),
]

# ============================================================
# 1. データ読み込み
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit_template = pd.read_csv('data/sample_submit.csv', header=None)

train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)
idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))

X_train_raw = train[spec_cols].values
X_test_raw  = test[spec_cols].values


def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s


# ============================================================
# 2. 学びの整理（自動分析）
# ============================================================
print("=" * 60)
print("📊 これまでの学びの整理")
print("=" * 60)
print("""
  ■ 確定した法則:
    1. OOF改善 ≠ LB改善（逆相関傾向）
    2. 特徴量の「足し算」→ 樹種暗記 → LB悪化
    3. Mixupは唯一LBを改善した手法 (12.615→11.80)
    4. α=0.5 > α=0.3 は失敗 → αは小さい方が良い
    5. Multi-seed平均は微悪化 → seed42単独が最良
    6. 水バンドのみは情報不足 (12.656)

  ■ 新仮説:
    → α=0.1~0.2 がさらに良い可能性
       (Beta(0.1,0.1)は95%以上が片親に極めて近い)
    → PLS(少成分) + Mixupが未試行の最有力候補
       (PLSは自然に樹種不変な潜在変数を抽出)
    → n_augment=200~300が最適かもしれない
       (500は実データ比率を下げすぎ？)
""")


# ============================================================
# 3. Mixup関数（11.80と完全同一）
# ============================================================
def mixup_augmentation(X, y, species, n_augment=500, alpha=0.3, seed=42):
    rng = np.random.RandomState(seed)
    unique_species = np.unique(species)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        sp1, sp2 = rng.choice(unique_species, size=2, replace=False)
        idx1 = rng.choice(np.where(species == sp1)[0])
        idx2 = rng.choice(np.where(species == sp2)[0])
        lam = rng.beta(alpha, alpha)
        X_aug.append(lam * X[idx1] + (1 - lam) * X[idx2])
        y_aug.append(lam * y[idx1] + (1 - lam) * y[idx2])
    return np.array(X_aug), np.array(y_aug)


def systematic_mixup(X, y, species, per_pair=5, alpha=0.3, seed=42):
    """全樹種ペアから均等にサンプル生成"""
    rng = np.random.RandomState(seed)
    unique_sp = np.unique(species)
    X_aug, y_aug = [], []
    for i, sp1 in enumerate(unique_sp):
        for sp2 in unique_sp[i+1:]:
            for _ in range(per_pair):
                i1 = rng.choice(np.where(species == sp1)[0])
                i2 = rng.choice(np.where(species == sp2)[0])
                lam = rng.beta(alpha, alpha)
                X_aug.append(lam * X[i1] + (1 - lam) * X[i2])
                y_aug.append(lam * y[i1] + (1 - lam) * y[i2])
    return np.array(X_aug), np.array(y_aug)


# ============================================================
# 4. 実験ランナー（LGB版）
# ============================================================
def run_lgb_experiment(name, alpha=0.3, n_augment=500,
                       use_systematic=False, per_pair=5,
                       lgb_override=None):
    """LGB実験（11.80のパイプラインをベースに最小変更）"""
    print(f"\n{'─'*55}")
    print(f"🔬 {name}")
    print(f"{'─'*55}")

    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred = np.zeros(len(train))
    fold_rmses = []

    lgb_params = dict(
        n_estimators=1000, learning_rate=0.03,
        max_depth=5, num_leaves=31,
        subsample=0.8, colsample_bytree=0.3,
        random_state=42, verbosity=-1
    )
    if lgb_override:
        lgb_params.update(lgb_override)

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups
    )):
        va_sp = train.iloc[va_idx]['樹種'].unique()
        tr_sp = groups.iloc[tr_idx].values
        X_tr = X_train_raw[tr_idx]
        y_tr = y_train_log.iloc[tr_idx].values
        X_va = X_train_raw[va_idx]
        y_va = y_train_log.iloc[va_idx].values

        # Mixup
        if use_systematic:
            X_mix, y_mix = systematic_mixup(
                X_tr, y_tr, tr_sp, per_pair=per_pair,
                alpha=alpha, seed=42 + fold)
        else:
            X_mix, y_mix = mixup_augmentation(
                X_tr, y_tr, tr_sp, n_augment=n_augment,
                alpha=alpha, seed=42 + fold)

        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix])
        y_aug = np.concatenate([y_tr, y_mix])

        if fold == 0:
            print(f"  orig={n_orig}, mixup={len(X_mix)}, total={len(X_aug)}")

        # 特徴量（11.80と完全同一）
        snv_aug = apply_snv(X_aug)
        d1_aug = savgol_filter(snv_aug, 15, 2, deriv=1, axis=1)
        snv_va = apply_snv(X_va)
        d1_va = savgol_filter(snv_va, 15, 2, deriv=1, axis=1)
        snv_te = apply_snv(X_test_raw)
        d1_te = savgol_filter(snv_te, 15, 2, deriv=1, axis=1)

        r_aug = (X_aug[:, idx_1940]/(X_aug[:, idx_1300]+1e-8)).reshape(-1,1)
        r_va = (X_va[:, idx_1940]/(X_va[:, idx_1300]+1e-8)).reshape(-1,1)
        r_te = (X_test_raw[:, idx_1940]/(X_test_raw[:, idx_1300]+1e-8)).reshape(-1,1)
        s_aug = np.std(X_aug, axis=1, keepdims=True)
        s_va = np.std(X_va, axis=1, keepdims=True)
        s_te = np.std(X_test_raw, axis=1, keepdims=True)

        # PCA (orig only)
        snv_orig = apply_snv(X_tr)
        pca = PCA(n_components=10, random_state=42)
        pca.fit(snv_orig)
        pc_aug = pca.transform(snv_aug)
        pc_va = pca.transform(snv_va)
        pc_te = pca.transform(snv_te)

        # KNN (orig only)
        pc_orig = pca.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='cosine')
        knn.fit(pc_orig)

        _, idx_k = knn.kneighbors(pc_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = idx_k[i]
            v = nb[nb != i][:5] if i < n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1, 1)

        _, idx_v = knn.kneighbors(pc_va, 5)
        knn_va = np.mean(y_tr[idx_v], axis=1).reshape(-1, 1)
        _, idx_t = knn.kneighbors(pc_te, 5)
        knn_te = np.mean(y_tr[idx_t], axis=1).reshape(-1, 1)

        ft = np.hstack([snv_aug, d1_aug, pc_aug, knn_aug, r_aug, s_aug])
        fv = np.hstack([snv_va, d1_va, pc_va, knn_va, r_va, s_va])
        fe = np.hstack([snv_te, d1_te, pc_te, knn_te, r_te, s_te])

        model = lgb.LGBMRegressor(**lgb_params)
        model.fit(ft, y_aug,
                  eval_set=[(fv, y_va)],
                  callbacks=[lgb.early_stopping(30, verbose=False)])

        pv = np.expm1(model.predict(fv))
        pt = np.expm1(model.predict(fe))
        oof_pred[va_idx] = pv
        final_pred += pt / 5

        rmse = np.sqrt(mean_squared_error(np.expm1(y_va), pv))
        fold_rmses.append(rmse)
        print(f"  Fold {fold+1} RMSE: {rmse:.4f}  ({list(va_sp)})")

    oof_rmse = np.sqrt(mean_squared_error(np.expm1(y_train_log), oof_pred))
    fm = np.mean(fold_rmses)
    fs = np.std(fold_rmses)
    print(f"  🌟 OOF: {oof_rmse:.4f}  Fold平均: {fm:.4f} ± {fs:.4f}")

    out = submit_template.copy()
    out[1] = np.clip(final_pred, 0, None)
    safe = name.replace(" ", "_").replace("(", "").replace(")", "").replace(",", "")
    fname = f'submission_{safe}.csv'
    out.to_csv(fname, index=False, header=False)
    print(f"  ✅ {fname}")

    return {'name': name, 'oof': oof_rmse, 'fold_mean': fm,
            'fold_std': fs, 'pred': final_pred, 'oof_pred': oof_pred,
            'fname': fname}


# ============================================================
# 5. PLS実験ランナー
# ============================================================
def run_pls_experiment(name, n_components=5, alpha=0.3, n_augment=500):
    """PLS回帰 + Mixup（ケモメトリクスの王道）"""
    print(f"\n{'─'*55}")
    print(f"🔬 {name} (PLS n_comp={n_components})")
    print(f"{'─'*55}")

    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred = np.zeros(len(train))
    fold_rmses = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups
    )):
        va_sp = train.iloc[va_idx]['樹種'].unique()
        tr_sp = groups.iloc[tr_idx].values
        X_tr = X_train_raw[tr_idx]
        y_tr = y_train_log.iloc[tr_idx].values
        X_va = X_train_raw[va_idx]
        y_va = y_train_log.iloc[va_idx].values

        # Mixup
        X_mix, y_mix = mixup_augmentation(
            X_tr, y_tr, tr_sp, n_augment=n_augment,
            alpha=alpha, seed=42 + fold)

        X_aug = np.vstack([X_tr, X_mix])
        y_aug = np.concatenate([y_tr, y_mix])

        # SNV前処理
        snv_aug = apply_snv(X_aug)
        snv_va = apply_snv(X_va)
        snv_te = apply_snv(X_test_raw)

        if fold == 0:
            print(f"  orig={len(X_tr)}, mixup={len(X_mix)}, "
                  f"total={len(X_aug)}, n_comp={n_components}")

        # PLS fit
        pls = PLSRegression(n_components=n_components, scale=False)
        pls.fit(snv_aug, y_aug)

        pv = np.expm1(pls.predict(snv_va).ravel())
        pt = np.expm1(pls.predict(snv_te).ravel())
        oof_pred[va_idx] = pv
        final_pred += pt / 5

        rmse = np.sqrt(mean_squared_error(np.expm1(y_va), pv))
        fold_rmses.append(rmse)
        print(f"  Fold {fold+1} RMSE: {rmse:.4f}  ({list(va_sp)})")

    oof_rmse = np.sqrt(mean_squared_error(np.expm1(y_train_log), oof_pred))
    fm = np.mean(fold_rmses)
    fs = np.std(fold_rmses)
    print(f"  🌟 OOF: {oof_rmse:.4f}  Fold平均: {fm:.4f} ± {fs:.4f}")

    out = submit_template.copy()
    out[1] = np.clip(final_pred, 0, None)
    safe = name.replace(" ", "_").replace("(", "").replace(")", "").replace(",", "")
    fname = f'submission_{safe}.csv'
    out.to_csv(fname, index=False, header=False)
    print(f"  ✅ {fname}")

    return {'name': name, 'oof': oof_rmse, 'fold_mean': fm,
            'fold_std': fs, 'pred': final_pred, 'oof_pred': oof_pred,
            'fname': fname}


# ============================================================
# 6. 全実験実行
# ============================================================
print("\n" + "=" * 60)
print("🧪 実験バッチ実行")
print("=" * 60)

results = []

# --- 6a. 11.80再現（ベースライン） ---
results.append(run_lgb_experiment(
    "G1_baseline_a03_n500", alpha=0.3, n_augment=500))

# --- 6b. α値の引き下げ ---
results.append(run_lgb_experiment(
    "G2_alpha015_n500", alpha=0.15, n_augment=500))

results.append(run_lgb_experiment(
    "G3_alpha02_n500", alpha=0.2, n_augment=500))

results.append(run_lgb_experiment(
    "G4_alpha01_n500", alpha=0.1, n_augment=500))

# --- 6c. 生成数の変更 ---
results.append(run_lgb_experiment(
    "G5_alpha03_n200", alpha=0.3, n_augment=200))

results.append(run_lgb_experiment(
    "G6_alpha03_n300", alpha=0.3, n_augment=300))

# --- 6d. Systematic mixup（全ペア均等） ---
results.append(run_lgb_experiment(
    "G7_systematic_5pp", alpha=0.3,
    use_systematic=True, per_pair=5))

# --- 6e. LGB正則化強化 + Mixup ---
results.append(run_lgb_experiment(
    "G8_depth3_n500", alpha=0.3, n_augment=500,
    lgb_override={'max_depth': 3, 'num_leaves': 15}))

results.append(run_lgb_experiment(
    "G9_depth4_n500", alpha=0.3, n_augment=500,
    lgb_override={'max_depth': 4, 'num_leaves': 20}))

# --- 6f. PLS + Mixup ---
for nc in [3, 5, 7, 10, 15]:
    results.append(run_pls_experiment(
        f"P{nc}_pls{nc}_a03", n_components=nc,
        alpha=0.3, n_augment=500))

# --- 6g. PLS + Mixup (α変更) ---
results.append(run_pls_experiment(
    "P5a015_pls5_a015", n_components=5,
    alpha=0.15, n_augment=500))


# ============================================================
# 7. 結果比較
# ============================================================
print(f"\n{'='*60}")
print("📊 今回の全実験結果")
print(f"{'='*60}")
print(f"  {'実験名':<28s} {'OOF':>8s} {'Fold平均':>8s} {'±':>7s}")
print(f"  {'─'*28} {'─'*8} {'─'*8} {'─'*7}")
for r in sorted(results, key=lambda x: x['oof']):
    print(f"  {r['name']:<28s} {r['oof']:>8.2f} "
          f"{r['fold_mean']:>8.2f} {r['fold_std']:>7.2f}")


# ============================================================
# 8. テスト予測の多様性分析
# ============================================================
print(f"\n{'='*60}")
print("📊 テスト予測の多様性（baselineとの差分）")
print(f"{'='*60}")
base = results[0]  # G1_baseline
print(f"  基準: {base['name']}")
print(f"  {'実験名':<28s} {'RMSD':>7s} {'mean_diff':>10s} {'corr':>7s}")
print(f"  {'─'*28} {'─'*7} {'─'*10} {'─'*7}")
for r in results[1:]:
    diff = r['pred'] - base['pred']
    rmsd = np.sqrt(np.mean(diff**2))
    corr = np.corrcoef(r['pred'], base['pred'])[0, 1]
    print(f"  {r['name']:<28s} {rmsd:>7.3f} {np.mean(diff):>+10.3f} "
          f"{corr:>7.4f}")


# ============================================================
# 9. LGB×PLSブレンド探索
# ============================================================
print(f"\n{'='*60}")
print("📊 LGB × PLS ブレンド探索")
print(f"{'='*60}")

lgb_results = [r for r in results if r['name'].startswith('G')]
pls_results = [r for r in results if r['name'].startswith('P')]

y_true = np.expm1(y_train_log)
best_blend_score = 999
best_blend_info = None
blend_list = []

for lr in lgb_results:
    for pr in pls_results:
        for w in np.arange(0.1, 1.0, 0.1):
            oof_blend = w * lr['oof_pred'] + (1 - w) * pr['oof_pred']
            rmse = np.sqrt(mean_squared_error(y_true, oof_blend))
            blend_list.append((lr['name'], pr['name'], w, rmse))
            if rmse < best_blend_score:
                best_blend_score = rmse
                best_blend_info = (lr, pr, w)

# Top 10 blends
blend_list.sort(key=lambda x: x[3])
print(f"  Top 10 LGB×PLS blends (OOF基準):")
print(f"  {'LGB':<24s} {'PLS':<24s} {'w':>5s} {'OOF':>8s}")
for lgb_n, pls_n, w, rmse in blend_list[:10]:
    print(f"  {lgb_n:<24s} {pls_n:<24s} {w:>5.1f} {rmse:>8.2f}")

# ベストブレンドの提出ファイル作成
if best_blend_info:
    lr, pr, w = best_blend_info
    blend_pred = np.clip(w * lr['pred'] + (1 - w) * pr['pred'], 0, None)
    out = submit_template.copy()
    out[1] = blend_pred
    fname_blend = (f'submission_blend_{lr["name"]}_{pr["name"]}'
                   f'_w{w:.1f}.csv')
    out.to_csv(fname_blend, index=False, header=False)
    print(f"\n  ベストブレンド: {lr['name']} × {w:.1f} + "
          f"{pr['name']} × {1-w:.1f}")
    print(f"  OOF RMSE: {best_blend_score:.4f}")
    print(f"  ✅ {fname_blend}")


# ============================================================
# 10. 全スコア一覧
# ============================================================
print(f"\n{'='*60}")
print("📌 全スコア比較（過去 + 今回）")
print(f"{'='*60}")
print(f"  {'手法':<36s} {'OOF':>8s} {'LB':>8s}")
print(f"  {'─'*36} {'─'*8} {'─'*8}")

for name, oof, lb in HISTORY:
    oof_s = f"{oof:.2f}" if oof is not None else "---"
    lb_s = f"{lb:.3f}"
    m = " ★BEST" if lb == 11.800 else (" ↓" if lb > 12.0 else "")
    print(f"  {name:<36s} {oof_s:>8s} {lb_s:>8s}{m}")

print(f"  {'─'*36} {'─'*8} {'─'*8}")
for r in sorted(results, key=lambda x: x['oof']):
    print(f"  今回: {r['name']:<30s} {r['oof']:>8.2f} {'???':>8s}")

if best_blend_info:
    lr, pr, w = best_blend_info
    blend_name = f"Blend({lr['name']}+{pr['name']})"
    print(f"  今回: {blend_name:<30s} {best_blend_score:>8.2f} {'???':>8s}")


# ============================================================
# 11. 提出判断ガイド
# ============================================================
print(f"\n{'='*60}")
print("🧭 提出判断ガイド")
print(f"{'='*60}")

# α別のOOF傾向
alpha_results = [r for r in results if r['name'].startswith('G')
                 and 'alpha' in r['name']]
if alpha_results:
    print(f"\n  ■ α値の影響:")
    for r in alpha_results:
        print(f"    {r['name']:<28s} OOF={r['oof']:.2f}")

# n_augment別
n_results = [r for r in results if 'n200' in r['name'] or
             'n300' in r['name'] or 'n500' in r['name']]
if n_results:
    print(f"\n  ■ 生成数の影響:")
    for r in n_results:
        print(f"    {r['name']:<28s} OOF={r['oof']:.2f}")

# PLS成分数別
if pls_results:
    print(f"\n  ■ PLS成分数の影響:")
    for r in sorted(pls_results, key=lambda x: x['oof']):
        print(f"    {r['name']:<28s} OOF={r['oof']:.2f}")

# 判断ロジック
print(f"""
  ============================================================
  ■ 核心の問い: OOFが悪い方がLBが良いのか？
  ============================================================

  過去のデータポイント:
    OOF 12.68 → LB 14.50  (物理特徴量53個)
    OOF 14.10 → LB 12.65  (元Blend)
    OOF 15.65 → LB 12.80  (PLS特徴量)
    OOF 17.21 → LB 12.62  (LGB単独)
    OOF 17.64 → LB 11.87  (SafeMulti3)
    OOF 17.82 → LB 12.33  (α=0.5)
    OOF 18.04 → LB 11.80  (Mixup α=0.3) ★BEST
    OOF 19.43 → LB 12.66  (水バンド)

  → 「OOF高い=LB良い」は必ずしも成立しない
  → ただし「OOF 17~18の範囲」が最もLBが良い傾向
  → OOF 19以上は情報不足で悪化
  → OOF 15以下は過学習で悪化

  ■ 今回のOOF結果に基づく推奨:
""")

# OOFが17~18の範囲にある実験を抽出
sweet_spot = [r for r in results if 17.0 <= r['oof'] <= 18.5]
sweet_spot.sort(key=lambda x: x['oof'])

if sweet_spot:
    print(f"  「OOF 17~18.5のスイートスポット」にある実験:")
    for i, r in enumerate(sweet_spot):
        marker = " ← 最優先" if i == 0 else ""
        print(f"    {r['name']:<28s} OOF={r['oof']:.2f} → {r['fname']}{marker}")
else:
    print(f"  スイートスポットに該当する実験なし")

# Baseline以外でOOFが最も近い実験
non_baseline = [r for r in results if r['name'] != 'G1_baseline_a03_n500']
if non_baseline:
    closest = min(non_baseline, key=lambda x: abs(x['oof'] - 18.04))
    print(f"\n  11.80(OOF=18.04)に最もOOFが近い実験:")
    print(f"    {closest['name']} (OOF={closest['oof']:.2f})")
    print(f"    → {closest['fname']}")

print(f"""
  ■ 推奨提出順:
""")

# 優先度付きリスト作成
priority = []

# 1. 11.80再現確認（まだ未確認の場合）
priority.append(("G1_baseline_a03_n500",
                 "11.80再現確認（未提出なら最優先）"))

# 2. αを下げた実験（新仮説の検証）
for r in results:
    if 'alpha01' in r['name'] or 'alpha015' in r['name']:
        priority.append((r['name'],
                         f"α引き下げの効果検証 (OOF={r['oof']:.2f})"))

# 3. PLS単独（全く異なるアプローチ）
best_pls = min(pls_results, key=lambda x: x['oof']) if pls_results else None
if best_pls:
    priority.append((best_pls['name'],
                     f"PLS単独 (OOF={best_pls['oof']:.2f})"))

# 4. ブレンド
if best_blend_info:
    priority.append((fname_blend,
                     f"LGB×PLSブレンド (OOF={best_blend_score:.2f})"))

for i, (name, reason) in enumerate(priority[:6]):
    print(f"    {i+1}. {name}")
    print(f"       理由: {reason}")


# ============================================================
# 12. 全提出ファイル一覧
# ============================================================
print(f"\n{'='*60}")
print("📁 全提出ファイル一覧")
print(f"{'='*60}")
for r in sorted(results, key=lambda x: x['oof']):
    print(f"  {r['fname']:<50s} OOF={r['oof']:.2f}")
if best_blend_info:
    print(f"  {fname_blend:<50s} OOF={best_blend_score:.2f} (blend)")


# ============================================================
# 13. 次回検討事項
# ============================================================
print(f"\n{'='*60}")
print("📝 次回検討事項")
print(f"{'='*60}")
print(f"""
  ■ 今回の結果次第で:

  α引き下げが改善:
    → α=0.05を追加テスト
    → 同時にn_augment=300と組み合わせ

  α引き下げが悪化:
    → α=0.3が最適と確定。別方向へ
    → MSC(Multiplicative Scatter Correction)前処理に切り替え
    → SNVの代わりにMSCを使用→散乱補正の質が変わる

  PLSが意外と良い:
    → PLS scores(潜在変数)をLGBの入力にする
    → PLS n_comp=3~5 のscoresだけで予測
    → 「樹種不変な潜在変数」のみを使うことで汎化

  PLSブレンドが改善:
    → LGB:PLS比率の精密探索(0.05刻み)
    → Ridge回帰も混ぜた3モデルブレンド

  ■ 根本的な方向転換（大幅改善が必要な場合）:
    - 1D-CNN: スペクトルの局所パターンを畳み込みで自動学習
    - Domain Adversarial Training: 樹種を区別できない特徴量を学習
    - 含水率の区間別モデル: 高含水率域と低含水率域で別モデル
""")

📊 これまでの学びの整理

  ■ 確定した法則:
    1. OOF改善 ≠ LB改善（逆相関傾向）
    2. 特徴量の「足し算」→ 樹種暗記 → LB悪化
    3. Mixupは唯一LBを改善した手法 (12.615→11.80)
    4. α=0.5 > α=0.3 は失敗 → αは小さい方が良い
    5. Multi-seed平均は微悪化 → seed42単独が最良
    6. 水バンドのみは情報不足 (12.656)

  ■ 新仮説:
    → α=0.1~0.2 がさらに良い可能性
       (Beta(0.1,0.1)は95%以上が片親に極めて近い)
    → PLS(少成分) + Mixupが未試行の最有力候補
       (PLSは自然に樹種不変な潜在変数を抽出)
    → n_augment=200~300が最適かもしれない
       (500は実データ比率を下げすぎ？)


🧪 実験バッチ実行

───────────────────────────────────────────────────────
🔬 G1_baseline_a03_n500
───────────────────────────────────────────────────────
  orig=940, mixup=500, total=1440
  Fold 1 RMSE: 7.9258  (['ウエンジ', 'トチ'])
  Fold 2 RMSE: 17.8934  (['チェリー', 'ヒノキ'])
  Fold 3 RMSE: 20.7025  (['ウォールナット', 'クリ'])
  Fold 4 RMSE: 19.7955  (['ナラ', 'ベイマツ', 'ホワイトオーク'])
  Fold 5 RMSE: 21.4266  (['イチョウ', 'スプルース', '米ヒバ'])
  🌟 OOF: 18.0379  Fold平均: 17.5488 ± 4.9548
  ✅ submission_G1_baseline_a03_n500.csv

───────────────────────────────────────────────────────
🔬 G2_alpha015_n500
─────────

In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 過去スコア記録
# ============================================================
HISTORY = [
    ("LGB単独(元特徴量)",          17.21, 12.615),
    ("元Blend(LGB/PLS/Ridge)",    14.10, 12.647),
    ("正則化強化(LGB単独)",        17.60, 12.760),
    ("Huber Loss",                17.53, 12.770),
    ("PLS予測を特徴量追加",        15.65, 12.800),
    ("逆距離加重KNN",             17.13, 12.940),
    ("d2(二次微分)追加",           16.12, 13.410),
    ("物理特徴量53個追加",         12.68, 14.500),
    ("Mixup(500,α=0.3)seed42",   18.04, 11.800),
    ("MultiSeed5+2w2000+3w1000", None,  12.249),
    ("SafeMultiSeed3(ensemble)", 17.64, 11.870),
    ("B_alpha05",                17.82, 12.326),
    ("E_water_bands_only",       19.43, 12.656),
    ("G2_alpha015_n500",         18.43, 11.935),
]

# ============================================================
# 確定パラメータ（これ以上変えない）
# ============================================================
CONFIRMED_ALPHA   = 0.3
CONFIRMED_N_AUG   = 500

print("=" * 60)
print("📊 確定した最適設定")
print("=" * 60)
print(f"  α = {CONFIRMED_ALPHA} (0.1~0.5全て検証済)")
print(f"  n_augment = {CONFIRMED_N_AUG}")
print(f"  seed = 42 (ただしseed依存性が高い → 探索の余地)")
print(f"  LGB: depth=5, leaves=31, lr=0.03")
print(f"  前処理: SNV + SG d1 + PCA10 + KNN5 + ratio + std")
print(f"  目標変換: log1p")
print(f"  ベイスギ除外")
print("=" * 60)

# ============================================================
# 1. データ読み込み
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit_template = pd.read_csv('data/sample_submit.csv', header=None)

train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)
idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))

X_train_raw = train[spec_cols].values
X_test_raw  = test[spec_cols].values


def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s


def mixup_augmentation(X, y, species, n_augment=500, alpha=0.3, seed=42):
    rng = np.random.RandomState(seed)
    unique_species = np.unique(species)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        sp1, sp2 = rng.choice(unique_species, size=2, replace=False)
        idx1 = rng.choice(np.where(species == sp1)[0])
        idx2 = rng.choice(np.where(species == sp2)[0])
        lam = rng.beta(alpha, alpha)
        X_aug.append(lam * X[idx1] + (1 - lam) * X[idx2])
        y_aug.append(lam * y[idx1] + (1 - lam) * y[idx2])
    return np.array(X_aug), np.array(y_aug)


# ============================================================
# 2. 11.80パイプライン完全再現関数
# ============================================================
def run_seed(seed, verbose=True):
    """seed1つでCV→テスト予測を返す（11.80パイプライン完全再現）"""
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred   = np.zeros(len(train))
    fold_rmses = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups
    )):
        tr_sp = groups.iloc[tr_idx].values
        X_tr = X_train_raw[tr_idx]
        y_tr = y_train_log.iloc[tr_idx].values
        X_va = X_train_raw[va_idx]
        y_va = y_train_log.iloc[va_idx].values

        # Mixup
        X_mix, y_mix = mixup_augmentation(
            X_tr, y_tr, tr_sp,
            n_augment=CONFIRMED_N_AUG,
            alpha=CONFIRMED_ALPHA,
            seed=seed + fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix])
        y_aug = np.concatenate([y_tr, y_mix])

        # SNV + d1
        snv_aug = apply_snv(X_aug)
        d1_aug  = savgol_filter(snv_aug, 15, 2, deriv=1, axis=1)
        snv_va  = apply_snv(X_va)
        d1_va   = savgol_filter(snv_va, 15, 2, deriv=1, axis=1)
        snv_te  = apply_snv(X_test_raw)
        d1_te   = savgol_filter(snv_te, 15, 2, deriv=1, axis=1)

        r_aug = (X_aug[:, idx_1940]/(X_aug[:, idx_1300]+1e-8)).reshape(-1,1)
        r_va  = (X_va[:, idx_1940]/(X_va[:, idx_1300]+1e-8)).reshape(-1,1)
        r_te  = (X_test_raw[:, idx_1940]/(X_test_raw[:, idx_1300]+1e-8)).reshape(-1,1)
        s_aug = np.std(X_aug, axis=1, keepdims=True)
        s_va  = np.std(X_va, axis=1, keepdims=True)
        s_te  = np.std(X_test_raw, axis=1, keepdims=True)

        # PCA
        snv_orig = apply_snv(X_tr)
        pca = PCA(n_components=10, random_state=42)
        pca.fit(snv_orig)
        pc_aug = pca.transform(snv_aug)
        pc_va  = pca.transform(snv_va)
        pc_te  = pca.transform(snv_te)

        # KNN
        pc_orig = pca.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='cosine')
        knn.fit(pc_orig)

        _, ik = knn.kneighbors(pc_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]
            v = nb[nb != i][:5] if i < n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1, 1)

        _, iv = knn.kneighbors(pc_va, 5)
        knn_va = np.mean(y_tr[iv], axis=1).reshape(-1, 1)
        _, it = knn.kneighbors(pc_te, 5)
        knn_te = np.mean(y_tr[it], axis=1).reshape(-1, 1)

        ft = np.hstack([snv_aug, d1_aug, pc_aug, knn_aug, r_aug, s_aug])
        fv = np.hstack([snv_va, d1_va, pc_va, knn_va, r_va, s_va])
        fe = np.hstack([snv_te, d1_te, pc_te, knn_te, r_te, s_te])

        model = lgb.LGBMRegressor(
            n_estimators=1000, learning_rate=0.03,
            max_depth=5, num_leaves=31,
            subsample=0.8, colsample_bytree=0.3,
            random_state=42, verbosity=-1)
        model.fit(ft, y_aug,
                  eval_set=[(fv, y_va)],
                  callbacks=[lgb.early_stopping(30, verbose=False)])

        pv = np.expm1(model.predict(fv))
        pt = np.expm1(model.predict(fe))
        oof_pred[va_idx] = pv
        final_pred += pt / 5

        rmse = np.sqrt(mean_squared_error(np.expm1(y_va), pv))
        fold_rmses.append(rmse)

    oof_rmse = np.sqrt(mean_squared_error(np.expm1(y_train_log), oof_pred))

    if verbose:
        print(f"  seed={seed:4d}  OOF={oof_rmse:.4f}  "
              f"Fold平均={np.mean(fold_rmses):.4f} ± {np.std(fold_rmses):.4f}")

    return {
        'seed': seed,
        'oof_rmse': oof_rmse,
        'fold_mean': np.mean(fold_rmses),
        'fold_std': np.std(fold_rmses),
        'fold_rmses': fold_rmses,
        'test_pred': final_pred,
        'oof_pred': oof_pred,
    }


# ============================================================
# 3. 実験A: Seed探索（seed42の周辺 + ランダム）
# ============================================================
print(f"\n{'='*60}")
print("🔬 実験A: Seed探索")
print("   seed=42が当たりなら近傍にも当たりがある可能性")
print(f"{'='*60}")

search_seeds = [
    # seed42の近傍
    40, 41, 42, 43, 44, 45,
    # やや離れた値
    30, 35, 50, 55,
    # 全く異なる値
    0, 7, 13, 21, 77, 99, 123, 256, 314, 777,
]

seed_results = []
for s in search_seeds:
    r = run_seed(s, verbose=True)
    seed_results.append(r)

# ソート
seed_results.sort(key=lambda x: x['oof_rmse'])

print(f"\n  --- Seed探索結果（OOF順）---")
print(f"  {'seed':>6s} {'OOF':>8s} {'Fold平均':>8s} {'±':>7s}")
print(f"  {'─'*6} {'─'*8} {'─'*8} {'─'*7}")
for r in seed_results:
    marker = " ← 11.80" if r['seed'] == 42 else ""
    print(f"  {r['seed']:>6d} {r['oof_rmse']:>8.4f} "
          f"{r['fold_mean']:>8.4f} {r['fold_std']:>7.4f}{marker}")

# seed42に最も近いOOFのseed（42以外）
best_non42 = [r for r in seed_results if r['seed'] != 42]
closest_to_42 = min(best_non42,
                    key=lambda x: abs(x['oof_rmse'] - 18.04))
print(f"\n  OOF≈18.04に最も近い別seed: "
      f"seed={closest_to_42['seed']} (OOF={closest_to_42['oof_rmse']:.4f})")

# テスト予測のseed間ばらつき分析
pred_42 = [r for r in seed_results if r['seed'] == 42][0]['test_pred']
print(f"\n  --- seed42との予測差分 ---")
print(f"  {'seed':>6s} {'RMSD':>7s} {'mean_diff':>10s} {'corr':>7s}")
for r in seed_results:
    if r['seed'] == 42:
        continue
    diff = r['test_pred'] - pred_42
    rmsd = np.sqrt(np.mean(diff**2))
    corr = np.corrcoef(r['test_pred'], pred_42)[0, 1]
    print(f"  {r['seed']:>6d} {rmsd:>7.3f} {np.mean(diff):>+10.3f} {corr:>7.4f}")


# ============================================================
# 4. 実験B: PLS scores → LGB ハイブリッド
# ============================================================
print(f"\n{'='*60}")
print("🔬 実験B: PLS scores → LGB ハイブリッド")
print("   PLSで樹種不変な潜在変数を抽出 → LGBで非線形予測")
print(f"{'='*60}")


def run_pls_lgb_hybrid(name, n_pls_comp=3, seed=42):
    """PLS scoresをLGBの入力特徴量に使うハイブリッド"""
    print(f"\n  🔬 {name} (PLS comp={n_pls_comp}, seed={seed})")

    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred   = np.zeros(len(train))
    fold_rmses = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups
    )):
        tr_sp = groups.iloc[tr_idx].values
        X_tr = X_train_raw[tr_idx]
        y_tr = y_train_log.iloc[tr_idx].values
        X_va = X_train_raw[va_idx]
        y_va = y_train_log.iloc[va_idx].values

        # Mixup
        X_mix, y_mix = mixup_augmentation(
            X_tr, y_tr, tr_sp,
            n_augment=CONFIRMED_N_AUG,
            alpha=CONFIRMED_ALPHA,
            seed=seed + fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix])
        y_aug = np.concatenate([y_tr, y_mix])

        # SNV
        snv_aug  = apply_snv(X_aug)
        snv_va   = apply_snv(X_va)
        snv_te   = apply_snv(X_test_raw)
        snv_orig = apply_snv(X_tr)

        # PLS（元データのみでfit → scores抽出）
        pls = PLSRegression(n_components=n_pls_comp, scale=False)
        pls.fit(snv_orig, y_tr)

        pls_scores_aug = pls.transform(snv_aug)
        pls_scores_va  = pls.transform(snv_va)
        pls_scores_te  = pls.transform(snv_te)

        # PLS予測値も特徴量に（ただし生予測ではなくscoreベース）
        pls_pred_aug = pls.predict(snv_aug).ravel().reshape(-1, 1)
        pls_pred_va  = pls.predict(snv_va).ravel().reshape(-1, 1)
        pls_pred_te  = pls.predict(snv_te).ravel().reshape(-1, 1)

        # d1
        d1_aug = savgol_filter(snv_aug, 15, 2, deriv=1, axis=1)
        d1_va  = savgol_filter(snv_va, 15, 2, deriv=1, axis=1)
        d1_te  = savgol_filter(snv_te, 15, 2, deriv=1, axis=1)

        # 従来特徴量
        r_aug = (X_aug[:, idx_1940]/(X_aug[:, idx_1300]+1e-8)).reshape(-1,1)
        r_va  = (X_va[:, idx_1940]/(X_va[:, idx_1300]+1e-8)).reshape(-1,1)
        r_te  = (X_test_raw[:, idx_1940]/(X_test_raw[:, idx_1300]+1e-8)).reshape(-1,1)
        s_aug = np.std(X_aug, axis=1, keepdims=True)
        s_va  = np.std(X_va, axis=1, keepdims=True)
        s_te  = np.std(X_test_raw, axis=1, keepdims=True)

        # PCA
        pca = PCA(n_components=10, random_state=42)
        pca.fit(snv_orig)
        pc_aug = pca.transform(snv_aug)
        pc_va  = pca.transform(snv_va)
        pc_te  = pca.transform(snv_te)

        # KNN
        pc_orig = pca.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='cosine')
        knn.fit(pc_orig)

        _, ik = knn.kneighbors(pc_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]
            v = nb[nb != i][:5] if i < n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1, 1)

        _, iv = knn.kneighbors(pc_va, 5)
        knn_va = np.mean(y_tr[iv], axis=1).reshape(-1, 1)
        _, it_ = knn.kneighbors(pc_te, 5)
        knn_te = np.mean(y_tr[it_], axis=1).reshape(-1, 1)

        # 特徴量結合: 元構成 + PLS scores + PLS pred
        ft = np.hstack([snv_aug, d1_aug, pc_aug, knn_aug, r_aug, s_aug,
                         pls_scores_aug, pls_pred_aug])
        fv = np.hstack([snv_va, d1_va, pc_va, knn_va, r_va, s_va,
                         pls_scores_va, pls_pred_va])
        fe = np.hstack([snv_te, d1_te, pc_te, knn_te, r_te, s_te,
                         pls_scores_te, pls_pred_te])

        if fold == 0:
            print(f"    feat_dim={ft.shape[1]} "
                  f"(元3123 + PLS_scores{n_pls_comp} + PLS_pred1 "
                  f"= +{n_pls_comp + 1})")

        model = lgb.LGBMRegressor(
            n_estimators=1000, learning_rate=0.03,
            max_depth=5, num_leaves=31,
            subsample=0.8, colsample_bytree=0.3,
            random_state=42, verbosity=-1)
        model.fit(ft, y_aug,
                  eval_set=[(fv, y_va)],
                  callbacks=[lgb.early_stopping(30, verbose=False)])

        pv = np.expm1(model.predict(fv))
        pt = np.expm1(model.predict(fe))
        oof_pred[va_idx] = pv
        final_pred += pt / 5

        rmse = np.sqrt(mean_squared_error(np.expm1(y_va), pv))
        fold_rmses.append(rmse)

    oof_rmse = np.sqrt(mean_squared_error(np.expm1(y_train_log), oof_pred))
    print(f"    🌟 OOF={oof_rmse:.4f} Fold平均={np.mean(fold_rmses):.4f}")

    return {
        'name': name, 'oof': oof_rmse,
        'fold_mean': np.mean(fold_rmses),
        'test_pred': final_pred, 'oof_pred': oof_pred,
    }


# PLS成分数を変えてテスト
hybrid_results = []
for nc in [2, 3, 5]:
    r = run_pls_lgb_hybrid(f"H_pls{nc}_lgb", n_pls_comp=nc, seed=42)
    hybrid_results.append(r)


# ============================================================
# 5. 実験C: PLS scores のみ → LGB（SNV+d1を除外）
# ============================================================
print(f"\n{'='*60}")
print("🔬 実験C: PLS scores のみ → LGB（次元大幅削減）")
print(f"{'='*60}")


def run_pls_scores_only(name, n_pls_comp=3, seed=42):
    """PLS scoresだけでLGB（3123次元→数十次元）"""
    print(f"\n  🔬 {name} (comp={n_pls_comp})")

    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred   = np.zeros(len(train))
    fold_rmses = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups
    )):
        tr_sp = groups.iloc[tr_idx].values
        X_tr = X_train_raw[tr_idx]
        y_tr = y_train_log.iloc[tr_idx].values
        X_va = X_train_raw[va_idx]
        y_va = y_train_log.iloc[va_idx].values

        # Mixup
        X_mix, y_mix = mixup_augmentation(
            X_tr, y_tr, tr_sp,
            n_augment=CONFIRMED_N_AUG,
            alpha=CONFIRMED_ALPHA,
            seed=seed + fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix])
        y_aug = np.concatenate([y_tr, y_mix])

        # SNV
        snv_orig = apply_snv(X_tr)
        snv_aug  = apply_snv(X_aug)
        snv_va   = apply_snv(X_va)
        snv_te   = apply_snv(X_test_raw)

        # PLS fit on orig only
        pls = PLSRegression(n_components=n_pls_comp, scale=False)
        pls.fit(snv_orig, y_tr)

        ps_aug = pls.transform(snv_aug)
        ps_va  = pls.transform(snv_va)
        ps_te  = pls.transform(snv_te)

        pp_aug = pls.predict(snv_aug).ravel().reshape(-1,1)
        pp_va  = pls.predict(snv_va).ravel().reshape(-1,1)
        pp_te  = pls.predict(snv_te).ravel().reshape(-1,1)

        # KNN on PLS scores
        ps_orig = pls.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='euclidean')
        knn.fit(ps_orig)

        _, ik = knn.kneighbors(ps_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]
            v = nb[nb != i][:5] if i < n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1, 1)

        _, iv = knn.kneighbors(ps_va, 5)
        knn_va = np.mean(y_tr[iv], axis=1).reshape(-1, 1)
        _, it_ = knn.kneighbors(ps_te, 5)
        knn_te = np.mean(y_tr[it_], axis=1).reshape(-1, 1)

        # ratio / std
        r_aug = (X_aug[:, idx_1940]/(X_aug[:, idx_1300]+1e-8)).reshape(-1,1)
        r_va  = (X_va[:, idx_1940]/(X_va[:, idx_1300]+1e-8)).reshape(-1,1)
        r_te  = (X_test_raw[:, idx_1940]/(X_test_raw[:, idx_1300]+1e-8)).reshape(-1,1)
        s_aug = np.std(X_aug, axis=1, keepdims=True)
        s_va  = np.std(X_va, axis=1, keepdims=True)
        s_te  = np.std(X_test_raw, axis=1, keepdims=True)

        # 特徴量: PLS scores + PLS pred + KNN + ratio + std
        ft = np.hstack([ps_aug, pp_aug, knn_aug, r_aug, s_aug])
        fv = np.hstack([ps_va, pp_va, knn_va, r_va, s_va])
        fe = np.hstack([ps_te, pp_te, knn_te, r_te, s_te])

        if fold == 0:
            print(f"    feat_dim={ft.shape[1]} (超コンパクト!)")

        model = lgb.LGBMRegressor(
            n_estimators=1000, learning_rate=0.03,
            max_depth=4, num_leaves=15,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, verbosity=-1)
        model.fit(ft, y_aug,
                  eval_set=[(fv, y_va)],
                  callbacks=[lgb.early_stopping(30, verbose=False)])

        pv = np.expm1(model.predict(fv))
        pt = np.expm1(model.predict(fe))
        oof_pred[va_idx] = pv
        final_pred += pt / 5

        rmse = np.sqrt(mean_squared_error(np.expm1(y_va), pv))
        fold_rmses.append(rmse)

    oof_rmse = np.sqrt(mean_squared_error(np.expm1(y_train_log), oof_pred))
    print(f"    🌟 OOF={oof_rmse:.4f} Fold平均={np.mean(fold_rmses):.4f}")

    return {
        'name': name, 'oof': oof_rmse,
        'fold_mean': np.mean(fold_rmses),
        'test_pred': final_pred, 'oof_pred': oof_pred,
    }


compact_results = []
for nc in [2, 3, 5, 7]:
    r = run_pls_scores_only(f"C_plsonly{nc}", n_pls_comp=nc, seed=42)
    compact_results.append(r)


# ============================================================
# 6. 最良seedの提出ファイル + 有望実験の提出
# ============================================================
print(f"\n{'='*60}")
print("📁 提出ファイル作成")
print(f"{'='*60}")

all_submissions = {}

# Seed探索のTop5
for r in seed_results[:5]:
    out = submit_template.copy()
    out[1] = np.clip(r['test_pred'], 0, None)
    fname = f"submission_seed{r['seed']}.csv"
    out.to_csv(fname, index=False, header=False)
    all_submissions[fname] = r['oof_rmse']
    print(f"  ✅ {fname} (OOF={r['oof_rmse']:.4f})")

# Hybridの全結果
for r in hybrid_results:
    out = submit_template.copy()
    out[1] = np.clip(r['test_pred'], 0, None)
    safe = r['name'].replace(" ", "_")
    fname = f"submission_{safe}.csv"
    out.to_csv(fname, index=False, header=False)
    all_submissions[fname] = r['oof']
    print(f"  ✅ {fname} (OOF={r['oof']:.4f})")

# Compactの全結果
for r in compact_results:
    out = submit_template.copy()
    out[1] = np.clip(r['test_pred'], 0, None)
    safe = r['name'].replace(" ", "_")
    fname = f"submission_{safe}.csv"
    out.to_csv(fname, index=False, header=False)
    all_submissions[fname] = r['oof']
    print(f"  ✅ {fname} (OOF={r['oof']:.4f})")

# seed42 × 最良compact のアンサンブル
y_true = np.expm1(y_train_log)
seed42_r = [r for r in seed_results if r['seed'] == 42][0]

best_blend_score = 999
best_blend_info = None

for cr in compact_results:
    for w in np.arange(0.5, 1.0, 0.05):
        oof_bl = w * seed42_r['oof_pred'] + (1-w) * cr['oof_pred']
        rmse_bl = np.sqrt(mean_squared_error(y_true, oof_bl))
        if rmse_bl < best_blend_score:
            best_blend_score = rmse_bl
            best_blend_info = (seed42_r, cr, w)

if best_blend_info:
    lr, cr, w = best_blend_info
    pred_bl = np.clip(w * lr['test_pred'] + (1-w) * cr['test_pred'], 0, None)
    out = submit_template.copy()
    out[1] = pred_bl
    fname_bl = f"submission_blend_seed42_x_{cr['name']}_w{w:.2f}.csv"
    out.to_csv(fname_bl, index=False, header=False)
    all_submissions[fname_bl] = best_blend_score
    print(f"  ✅ {fname_bl} (OOF={best_blend_score:.4f})")
    print(f"     seed42 × {w:.2f} + {cr['name']} × {1-w:.2f}")


# ============================================================
# 7. 全結果比較 + 提出ガイド
# ============================================================
print(f"\n{'='*60}")
print("📌 全スコア比較（過去 + 今回）")
print(f"{'='*60}")
print(f"  {'手法':<38s} {'OOF':>8s} {'LB':>8s}")
print(f"  {'─'*38} {'─'*8} {'─'*8}")

for name, oof, lb in HISTORY:
    oof_s = f"{oof:.2f}" if oof is not None else "---"
    m = " ★BEST" if lb == 11.800 else (" ↓" if lb > 12.0 else "")
    print(f"  {name:<38s} {oof_s:>8s} {lb:.3f}{m}")

print(f"  {'─'*38} {'─'*8} {'─'*8}")

# Seed探索結果
for r in seed_results[:5]:
    name = f"SeedSearch: seed={r['seed']}"
    marker = " (=11.80)" if r['seed'] == 42 else ""
    print(f"  {name:<38s} {r['oof_rmse']:>8.2f} {'???':>8s}{marker}")

# Hybrid結果
for r in hybrid_results:
    print(f"  {r['name']:<38s} {r['oof']:>8.2f} {'???':>8s}")

# Compact結果
for r in compact_results:
    print(f"  {r['name']:<38s} {r['oof']:>8.2f} {'???':>8s}")

# Blend
if best_blend_info:
    lr, cr, w = best_blend_info
    bname = f"Blend(seed42+{cr['name']})"
    print(f"  {bname:<38s} {best_blend_score:>8.2f} {'???':>8s}")


# ============================================================
# 8. 提出判断
# ============================================================
print(f"\n{'='*60}")
print("🧭 提出判断ガイド")
print(f"{'='*60}")

# Seed間のばらつきからLBの期待分布を推定
seed_oofs = [r['oof_rmse'] for r in seed_results]
seed_mean_oof = np.mean(seed_oofs)
seed_std_oof  = np.std(seed_oofs)
print(f"\n  ■ Seed探索の知見:")
print(f"    OOF平均={seed_mean_oof:.4f} ± {seed_std_oof:.4f}")
print(f"    OOF範囲: {min(seed_oofs):.4f} ~ {max(seed_oofs):.4f}")
print(f"    → seed依存性: {'高い' if seed_std_oof > 0.5 else '低い'}")

# Sweet spot分析
sweet = [r for r in seed_results
         if 17.8 <= r['oof_rmse'] <= 18.3 and r['seed'] != 42]
if sweet:
    print(f"\n  ■ OOF 17.8~18.3 のseed（42以外）:")
    for r in sweet:
        print(f"    seed={r['seed']}: OOF={r['oof_rmse']:.4f}")
    print(f"    → これらは11.80前後のLBが期待できる")

print(f"""
  ■ 推奨提出順:
""")

priority_list = []

# 1. Sweet spot seed
if sweet:
    best_sweet = min(sweet, key=lambda x: abs(x['oof_rmse'] - 18.04))
    priority_list.append((
        f"submission_seed{best_sweet['seed']}.csv",
        f"OOF={best_sweet['oof_rmse']:.4f}, 18.04に最も近い別seed"
    ))

# 2. Compact PLS (全く異なるアプローチ)
if compact_results:
    # OOFが18前後のcompact
    sweet_compact = [r for r in compact_results
                     if 17.0 <= r['oof'] <= 20.0]
    if sweet_compact:
        bc = min(sweet_compact, key=lambda x: abs(x['oof'] - 18.04))
        priority_list.append((
            f"submission_{bc['name']}.csv",
            f"OOF={bc['oof']:.4f}, PLS scores のみ→超コンパクト"
        ))

# 3. Blend
if best_blend_info:
    priority_list.append((
        fname_bl,
        f"OOF={best_blend_score:.4f}, seed42×PLS compact ブレンド"
    ))

for i, (fname, reason) in enumerate(priority_list[:3]):
    print(f"    {i+1}. {fname}")
    print(f"       理由: {reason}")

print(f"""
  ■ 残された大きな方向転換:
    - MSC前処理（SNVの代替）
    - 1D-CNN（PyTorchで10エポック程度の軽量版）
    - Domain Adversarial Training
    - Quantile Regression → 信頼区間も出力
""")

📊 確定した最適設定
  α = 0.3 (0.1~0.5全て検証済)
  n_augment = 500
  seed = 42 (ただしseed依存性が高い → 探索の余地)
  LGB: depth=5, leaves=31, lr=0.03
  前処理: SNV + SG d1 + PCA10 + KNN5 + ratio + std
  目標変換: log1p
  ベイスギ除外

🔬 実験A: Seed探索
   seed=42が当たりなら近傍にも当たりがある可能性
  seed=  40  OOF=17.6287  Fold平均=17.1481 ± 5.1035
  seed=  41  OOF=17.9259  Fold平均=17.3821 ± 5.3064
  seed=  42  OOF=18.0379  Fold平均=17.5488 ± 4.9548
  seed=  43  OOF=17.3682  Fold平均=16.8037 ± 5.0113
  seed=  44  OOF=17.9897  Fold平均=17.5082 ± 5.2317
  seed=  45  OOF=17.7613  Fold平均=17.2348 ± 5.0988
  seed=  30  OOF=17.8237  Fold平均=17.4057 ± 4.5680
  seed=  35  OOF=16.7930  Fold平均=16.4428 ± 4.1069
  seed=  50  OOF=17.9759  Fold平均=17.4224 ± 5.2171
  seed=  55  OOF=17.4177  Fold平均=16.8399 ± 5.0895
  seed=   0  OOF=17.7670  Fold平均=17.3156 ± 4.6724
  seed=   7  OOF=18.3663  Fold平均=17.8603 ± 5.2338
  seed=  13  OOF=17.8786  Fold平均=17.3431 ± 5.1403
  seed=  21  OOF=17.7410  Fold平均=17.2409 ± 4.9744
  seed=  77  OOF=17.8969  Fold平均=17.3755 ± 5.1090
  seed=  